# 🎨 27 · Generative models: diffusion and flow matching
### Scores and Langevin dynamics, denoising diffusion (DDPM), flow matching, classifier-free guidance and how to evaluate generated samples — on a 2-D differentiation fork, simulated single cells and a noised protein backbone

**BiologyToML** · Level 2 · Research frontier · Part F — Modern machine-learning building blocks · Module 27 of 44

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/xulabs/edu/blob/main/ml/tutorial/BiologyToML/notebooks/27_diffusion_flow_matching.ipynb)

| | |
|---|---|
| **Hardware target** | Google Colab free tier — CPU only (≈ 1–2 min to run everything); no GPU needed |
| **Core frameworks** | Python 3, NumPy, SciPy, PyTorch (CPU), scikit-learn, pandas, Matplotlib |
| **Prerequisites** | Level 1 (especially Modules 13, 21 and 22) and [Module 25](25_variational_inference_vae.ipynb) (latent-variable generative models, KL divergence, the reparameterisation trick) |
| **Estimated time** | 180 min (reading + running + the ✏️ exercises) |
| **Builds on** | [13 · Mixture models and EM](13_mixture_models_em.ipynb), [14 · PCA](14_pca_dimensionality_reduction.ipynb), [21 · Bayesian learning and MCMC](21_bayesian_learning_mcmc.ipynb), [22 · Neural networks](22_neural_networks.ipynb), [25 · Variational inference and VAEs](25_variational_inference_vae.ipynb), [26 · Attention and transformers](26_attention_transformers.ipynb) |
| **Research frontier covered** | denoising diffusion, score-based models, flow matching, conditional generation and guidance; generative models for proteins (RFdiffusion, AlphaFold 3) and cells — see [`FRONTIER_MAP.md`](../FRONTIER_MAP.md) |

> ⚠️ **Important notes.** For research and educational purposes only, not to be used for clinical diagnosis.
> All data are **SIMULATED**, so that every generated sample can be checked against a known truth: a 2-D
> "differentiation fork" written in this notebook (`sample_fork`: a progenitor stem and two fates with known shares
> 50 / 30 / 20 %), a two-state Gaussian mixture with an exact score, PBMC-like single-cell counts from `simulate_scrna`
> (six known cell types and their marker genes) and the Cα trace of an idealised four-helix bundle from
> `simulate_helix_bundle`. Real generative modelling works with far larger data (millions of cells, the whole PDB), far
> larger networks and GPUs; the algorithms are the same ones you implement here — see 🧭 *From toy to real data*.

## 🎯 Learning objectives

By the end of this notebook you will be able to:

1. **Explain** what a generative model must do (sample, condition, ideally evaluate a density) and **place** mixture models, VAEs, GANs, normalising flows, diffusion and flow matching on that map, using the change-of-variables formula.
2. **Define** the score $\nabla_x\log p(x)$, **compute** it for a Gaussian mixture, **sample** with Langevin dynamics, and **show** why noise-annealed scores fix the failure of plain Langevin sampling between separated modes.
3. **Derive** the closed form $q(x_t\mid x_0)=\mathcal N(\sqrt{\bar\alpha_t}\,x_0,(1-\bar\alpha_t)I)$ of the forward diffusion and **verify** it against the step-by-step Markov chain.
4. **Train** a denoising diffusion model (ε-prediction) in PyTorch, **sample** from it by ancestral sampling, and **verify** against an analytic answer that the denoiser has learned the score.
5. **Implement** flow matching, **integrate** its ODE with Euler steps, and **compare** sample quality against DDPM as a function of the number of network evaluations.
6. **Build** a class-conditional generator of single cells with label dropout and classifier-free guidance, and **quantify** the fidelity–diversity trade-off against a classifier, precision/recall and marker genes.
7. **Evaluate** generated samples honestly — energy distance, off-manifold fraction, branch proportions, nearest-neighbour memorisation tests — and **explain** why a perfect fit to the training set is a memoriser.
8. **Explain** how the same ideas generate protein backbones, complexes and docked poses (RFdiffusion, Chroma, AlphaFold 3, DiffDock) and why 3-D generation needs rotation equivariance.

## 🗺️ Contents

| Section | What you will do | ⏱ |
|---|---|---|
| 0 · Setup | imports, PyTorch, course toolkit | 5 min |
| 1 · The generative-modelling landscape | the fork; a mixture model as a sampler; yardsticks; change of variables | 20 min |
| 2 · Scores and Langevin dynamics | exact scores; Langevin sampling; why annealing over noise levels helps | 20 min |
| 3 · Forward diffusion | noise schedules; derive and verify q(x_t | x_0); the fork dissolves | 15 min |
| 4 · Learning to denoise (DDPM) | ε-prediction network; ancestral sampling; 🧪 score check, quality, memorisation | 30 min |
| 5 · Flow matching | velocity regression; Euler sampling; 🧪 quality vs number of steps; straight paths | 20 min |
| 6 · Conditional generation and guidance | generate cells of a requested type; 🧪 fidelity vs diversity | 25 min |
| 7 · Diffusion beyond points | noising a protein backbone; masking diffusion for sequences; RFdiffusion, AF3 | 15 min |
| 8 · Evaluation and pitfalls | 🧪 the perfect memoriser; metrics; compute cost | 15 min |
| 9 · 🧭 From toy to real data | protein and cell generators, data, GPUs, what goes wrong first | 5 min |
| ✅ / 🧠 / 🚀 / 📝 | takeaways, checkpoint, discussion, going further, lab note | 10 min |

## 🧫 The biological question

Most of the machine learning in this course *recognises*: it predicts a label, a value or a cluster for an example that
already exists. Two of the most ambitious goals of current computational biology need models that *create* instead:

* **Protein design.** Can we write down the backbone of a protein that has never existed — a binder that grips a viral
  surface protein, an enzyme pocket around a new substrate — and then find a sequence that folds into it? In 2023,
  RFdiffusion (Watson *et al.* 2023) generated such backbones by starting from random 3-D coordinates and *denoising*
  them step by step; many designs were confirmed experimentally. In 2024, AlphaFold 3 (Abramson *et al.* 2024) replaced
  the structure module of AlphaFold 2 with the same kind of denoising process to predict structures of whole complexes.
* **Virtual cells.** Can we predict what a cell type we have rarely measured looks like, or how a cell would respond to
  a perturbation nobody has tried? That requires sampling realistic new expression profiles from a learned
  distribution, conditioned on what we want.

Both problems are instances of **generative modelling**: learn a probability distribution $p(x)$ from examples, then draw
new examples from it — and from conditional versions $p(x\mid y)$. Module 25 built one generative model, the VAE. This
module builds the two families that now dominate: **diffusion models**, which learn to reverse a process that gradually
turns data into noise, and **flow matching**, which learns a velocity field that carries noise to data.

We do it on SIMULATED data with known truth, so every claim can be checked: a 2-D *differentiation fork* (progenitor
cells branching into two fates, like a low-dimensional embedding of haematopoiesis — Module 32 studies such
trajectories), PBMC-like single cells with six known types (a design modelled on the interferon-β PBMC experiment of
Kang *et al.* 2018), and the backbone of an idealised four-helix-bundle protein (the structure of Modules 26, 37 and 38).

---
## 0 · Environment setup

Run the cells **in order, from the top** (Shift + Enter). The first cell reports your hardware; this notebook uses PyTorch (preinstalled on Colab); a CPU is enough.

In [ ]:
# ── Hardware check ──
import platform
import subprocess
import sys

print("Python", sys.version.split()[0], "on", platform.system())
try:
    gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                         capture_output=True, text=True, timeout=10).stdout.strip()
    print("GPU:", gpu or "none detected — that is fine, a CPU is enough for this course.")
except (FileNotFoundError, subprocess.TimeoutExpired):
    print("GPU: none detected — that is fine, a CPU is enough for this course.")

In [ ]:
# ── Dependencies (Colab already has these; we install only what is missing) ──
import importlib

for module, pip_name in [("numpy", "numpy"), ("scipy", "scipy"), ("pandas", "pandas"), ("matplotlib", "matplotlib"), ("sklearn", "scikit-learn"), ("torch", "torch")]:
    try:
        importlib.import_module(module)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name], check=True)
print("✅ All dependencies are available.")

In [ ]:
# ── Global imports and configuration ──
import math
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from scipy.spatial.distance import cdist
from scipy.special import logsumexp as lse
from sklearn.linear_model import LogisticRegression
from sklearn.mixture import GaussianMixture

SEED = 0                                   # fixing the random seed makes every run identical
rng = np.random.default_rng(SEED)
plt.rcParams.update({"figure.dpi": 90, "figure.figsize": (6.4, 3.6), "axes.titlesize": 10,
                     "axes.labelsize": 9, "legend.fontsize": 8, "axes.spines.top": False,
                     "axes.spines.right": False})
print("NumPy", np.__version__, "| seed", SEED)

In [ ]:
# ── 0.1 PyTorch configuration and colours ──
torch.manual_seed(SEED)                  # PyTorch has its own random-number generator
torch.set_num_threads(1)                 # tiny networks: one thread is as fast and avoids slow-downs on shared CPUs
OKABE_ITO = ["#0072B2", "#D55E00", "#009E73", "#E69F00", "#CC79A7", "#56B4E9", "#F0E442"]   # colour-blind-safe
print("PyTorch", torch.__version__, "| threads:", torch.get_num_threads())

### 🧰 Course toolkit

The next cell copies the shared helpers this notebook needs from [`course_toolkit.py`](../course_toolkit.py).
`simulate_scrna` generates PBMC-like UMI counts with known cell types and marker genes; `log_normalize` and `pca` turn them into a 20-dimensional cell-state space; `plot_embedding` draws cells in 2-D; `simulate_helix_bundle` gives the Cα coordinates of a small protein and `AMINO_ACIDS` the 20-letter alphabet. Just run it.

In [ ]:
# ── Course toolkit (identical to ../course_toolkit.py v2.0; copied so this notebook runs alone) ──
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

MITO_GENES = ["MT-ND1", "MT-ND2", "MT-CO1", "MT-CO2", "MT-ATP8", "MT-ATP6", "MT-CO3", "MT-ND3",
              "MT-ND4L", "MT-ND4", "MT-ND5", "MT-ND6", "MT-CYB"]


INTERFERON_GENES = ["ISG15", "IFI6", "IFIT1", "IFIT3", "MX1", "OAS1", "IFI44L", "RSAD2", "IRF7", "STAT1"]


CYCLING_GENES = ["MKI67", "TOP2A", "CCNB1", "CDK1", "UBE2C", "BIRC5", "TYMS", "PCNA"]


PBMC_TYPES = {
    "CD4 T": ("lymphoid", ["CD3E", "CD3D", "IL7R", "CD4"], 0.5),
    "CD8 T": ("lymphoid", ["CD3E", "CD3D", "CD8A", "CD8B"], 0.5),
    "NK": ("lymphoid", ["NKG7", "GNLY", "KLRD1", "PRF1"], 2.2),
    "B": ("lymphoid", ["MS4A1", "CD79A", "CD79B", "CD19"], 2.4),
    "Monocyte": ("myeloid", ["CD14", "LYZ", "S100A8", "S100A9"], 2.6),
    "Dendritic": ("myeloid", ["FCER1A", "CST3", "CLEC10A", "HLA-DQA1"], 2.2),
}


def sample_negative_binomial(rng: np.random.Generator, mu, theta):
    """Negative-binomial counts with mean ``mu`` and inverse dispersion ``theta``.

    Variance = mu + mu²/theta (large theta → Poisson).  Drawn as a Gamma–Poisson mixture:
    each count is Poisson with a rate that itself varies from cell to cell.
    """
    mu = np.asarray(mu, float)
    theta = np.broadcast_to(np.asarray(theta, float), mu.shape)
    return rng.poisson(rng.gamma(theta, np.where(mu > 0, mu, 0.0) / theta)).astype(np.int32)


def make_expression_programs(rng: np.random.Generator, n_genes: int, cell_types: dict,
                             n_extra_markers: int = 20, extra_names=()):
    """Log-scale expression profiles (one row per cell type) for a simulated tissue.

    ``cell_types`` maps a type name to (lineage, named marker genes, marker strength).  Every
    type gets its named markers plus ``n_extra_markers`` anonymous ones; types of the same
    lineage share 20 lineage genes.  ``extra_names`` are reserved gene names (placed after the
    markers) that the caller gives its own behaviour.  Returns ``(log_profiles, genes, marker_of)``.
    """
    named = []
    for _, markers, _ in cell_types.values():
        named += [g for g in markers if g not in named]
    reserved = named + [g for g in extra_names if g not in named]
    if len(reserved) > n_genes:
        raise ValueError("n_genes is too small for the requested marker genes")
    genes = np.array(reserved + [f"Gene{j:04d}" for j in range(1, n_genes - len(reserved) + 1)], dtype=object)
    free = np.arange(len(reserved), n_genes)                       # anonymous genes available
    base = rng.normal(0.0, 2.0, n_genes)                           # gene-specific baseline (log scale)
    base[:len(named)] -= 2.0                                       # named markers are low elsewhere
    profiles = np.tile(base, (len(cell_types), 1)) + rng.normal(0, 0.15, (len(cell_types), n_genes))
    marker_of = np.array([""] * n_genes, dtype=object)
    lineages = sorted({v[0] for v in cell_types.values()})
    lineage_genes = {lin: rng.choice(free, 20, replace=False) for lin in lineages}
    for t, (name, (lineage, markers, strength)) in enumerate(cell_types.items()):
        profiles[t, lineage_genes[lineage]] += rng.normal(1.2, 0.3, 20)
        idx = [int(np.flatnonzero(genes == g)[0]) for g in markers]
        profiles[t, idx] += strength + 2.0 + rng.normal(0, 0.2, len(idx))
        extra = rng.choice(free, n_extra_markers, replace=False)
        profiles[t, extra] += rng.normal(0.6 + strength / 2, 0.3, n_extra_markers)
        for j in list(idx) + list(extra):
            marker_of[j] = name if marker_of[j] == "" else marker_of[j] + "/" + name
    return profiles, genes, marker_of


def simulate_scrna(rng: np.random.Generator, n_cells: int = 3000, n_genes: int = 1000, cell_types=None,
                   proportions=None, n_donors: int = 6, n_batches: int = 2, confounded: bool = False,
                   depth: float = 1500, batch_effect: float = 0.25, donor_effect: float = 0.15,
                   stim_effect: float = 1.0, theta: float = 8.0, doublet_rate: float = 0.0,
                   damaged_rate: float = 0.0, cycling_rate: float = 0.0, drop_type_in_batch=None) -> dict:
    """Simulate single-cell RNA-seq UMI counts for a PBMC-like sample collection.

    Design: ``n_donors`` donors, the first half *control* and the second half *stimulated*
    (an interferon response in every cell type plus a monocyte-specific response).  Donor
    ``d`` is sequenced in batch ``d % n_batches`` — or, with ``confounded=True``, every control
    donor in batch 1 and every stimulated donor in batch 2 (a design flaw to learn from).
    Counts are negative binomial around ``depth`` UMIs per cell.  Optional artefacts for
    quality control: doublets, damaged cells (high mitochondrial fraction), cycling cells.
    ``drop_type_in_batch=("B", 1)`` removes B cells from batch 1 (index 1, i.e. "batch2").

    Expression probabilities are renormalised within each cell (a fixed-size library, as in
    sequencing), so in stimulated cells the genes with ``true_logfc == 0`` lose a few per cent of
    their share (about −0.04 on the natural-log scale) to the induced genes: a compositional
    effect that is also present in real data.

    Returns a dict with ``counts`` (cells × genes, int32), ``obs`` (per-cell table with the
    true cell type, donor, batch, condition, …), ``var`` (per-gene table), ``true_logfc``
    (cell type × gene stimulation effect, natural log) and ``log_profiles``.
    """
    cell_types = PBMC_TYPES if cell_types is None else cell_types
    names = list(cell_types)
    T = len(names)
    p = np.full(T, 1 / T) if proportions is None else np.asarray(proportions, float) / np.sum(proportions)
    extra = MITO_GENES + INTERFERON_GENES + CYCLING_GENES
    profiles, genes, marker_of = make_expression_programs(rng, n_genes, cell_types, extra_names=extra)
    G = len(genes)
    is_mito, is_ifn, is_cyc = (np.isin(genes, MITO_GENES), np.isin(genes, INTERFERON_GENES),
                               np.isin(genes, CYCLING_GENES))
    profiles[:, is_mito] = rng.normal(3.0, 0.3, is_mito.sum())          # ≈ 4 % of UMIs are mitochondrial
    profiles[:, is_ifn | is_cyc] = rng.normal(-1.0, 0.4, (T, (is_ifn | is_cyc).sum()))
    # stimulation: interferon genes up everywhere + 40 monocyte-specific responders (up or down)
    true_logfc = np.zeros((T, G))
    true_logfc[:, is_ifn] = stim_effect * rng.normal(1.6, 0.3, is_ifn.sum())
    if "Monocyte" in names:
        responders = rng.choice(np.flatnonzero((marker_of == "") & ~is_mito & ~is_ifn & ~is_cyc), 40, replace=False)
        true_logfc[names.index("Monocyte"), responders] = stim_effect * rng.choice([-1, 1], 40) * rng.normal(1.2, 0.3, 40)
    # donors, batches, conditions
    donor = rng.integers(0, n_donors, n_cells)
    stimulated = donor >= n_donors // 2
    batch = stimulated.astype(int) if confounded else donor % n_batches
    batch = np.minimum(batch, n_batches - 1)
    cell_type = rng.choice(T, n_cells, p=p)
    if drop_type_in_batch is not None:
        t_drop, b_drop = names.index(drop_type_in_batch[0]), drop_type_in_batch[1]
        bad = (cell_type == t_drop) & (batch == b_drop)
        cell_type[bad] = rng.choice(np.delete(np.arange(T), t_drop), bad.sum(), p=np.delete(p, t_drop) / (1 - p[t_drop]))
    batch_shift = rng.normal(0, batch_effect, (n_batches, G))
    batch_depth = np.exp(rng.normal(0, 0.3, n_batches))
    donor_shift = rng.normal(0, donor_effect, (n_donors, G))
    log_rate = (profiles[cell_type] + batch_shift[batch] + donor_shift[donor]
                + true_logfc[cell_type] * stimulated[:, None])
    cycling = rng.random(n_cells) < cycling_rate
    log_rate[np.ix_(cycling, np.flatnonzero(is_cyc))] += 3.0
    damaged = rng.random(n_cells) < damaged_rate
    log_rate[np.ix_(damaged, np.flatnonzero(is_mito))] += 2.2
    lib = depth * batch_depth[batch] * np.exp(rng.normal(0, 0.35, n_cells))
    lib[damaged] *= 0.35
    prob = np.exp(log_rate - log_rate.max(axis=1, keepdims=True))
    prob /= prob.sum(axis=1, keepdims=True)
    theta_g = theta * np.exp(rng.normal(0, 0.5, G))
    mu = lib[:, None] * prob
    doublet = rng.random(n_cells) < doublet_rate
    partner = rng.integers(0, n_cells, n_cells)
    mu[doublet] = 0.9 * mu[doublet] + 0.9 * mu[partner[doublet]]       # two cells in one droplet
    counts = sample_negative_binomial(rng, mu, theta_g)
    obs = pd.DataFrame({
        "cell_type": np.where(doublet, "Doublet", np.array(names, dtype=object)[cell_type]),
        "true_type": np.array(names, dtype=object)[cell_type],
        "lineage": np.array([cell_types[n][0] for n in names], dtype=object)[cell_type],
        "donor": [f"donor{d + 1}" for d in donor],
        "condition": np.where(stimulated, "stimulated", "control"),
        "batch": [f"batch{b + 1}" for b in batch],
        "n_counts": counts.sum(axis=1),
        "n_genes": (counts > 0).sum(axis=1),
        "pct_mito": 100 * counts[:, is_mito].sum(axis=1) / np.maximum(counts.sum(axis=1), 1),
        "is_doublet": doublet, "is_damaged": damaged, "is_cycling": cycling,
    }, index=[f"cell{i:05d}" for i in range(n_cells)])
    var = pd.DataFrame({"marker_of": marker_of, "is_mito": is_mito, "is_interferon": is_ifn,
                        "is_cycling": is_cyc, "theta": theta_g}, index=genes)
    return dict(counts=counts, obs=obs, var=var, log_profiles=pd.DataFrame(profiles, index=names, columns=genes),
                true_logfc=pd.DataFrame(true_logfc, index=names, columns=genes))


AMINO_ACIDS = "ACDEFGHIKLMNPQRSTVWY"


def simulate_helix_bundle(n_helices: int = 4, helix_length: int = 14, loop_length: int = 4,
                          rng=None, noise: float = 0.0) -> dict:
    """Cα coordinates (Å) of an idealised four-helix-bundle protein, its contact map and secondary structure.

    Helices are ideal right-handed α-helices (3.6 residues per turn, 1.5 Å rise, 2.3 Å radius)
    packed antiparallel on the corners of a 10 Å square and joined by short loops.  ``noise`` adds
    Gaussian jitter (needs ``rng``).  Contacts: Cα–Cα distance < 8 Å and |i − j| ≥ 3.

    Returns a dict with ``coords`` (L × 3), ``ss`` (string of 'H'/'L'), ``distances`` and ``contacts``.
    """
    corners = 5.0 * np.array([[1, 1], [1, -1], [-1, -1], [-1, 1]], float)
    pts, ss = [], ""
    for h in range(n_helices):
        cx, cy = corners[h % 4] * (1 + h // 4)
        up = h % 2 == 0
        k = np.arange(helix_length)
        angle = np.deg2rad(100.0) * k * (1 if up else -1)      # turn with the direction of travel, so
        z = 1.5 * (k - (helix_length - 1) / 2) * (1 if up else -1)   # every helix is right-handed
        helix = np.c_[cx + 2.3 * np.cos(angle), cy + 2.3 * np.sin(angle), z]
        if pts:                                                      # loop: circular arc, 3.8 Å steps
            start, end = pts[-1], helix[0]
            d, m = np.linalg.norm(end - start), loop_length + 1
            lo, hi = 1e-6, 2 * np.pi - 1e-6                              # arc angle Θ with m equal chords
            for _ in range(60):
                theta = (lo + hi) / 2
                lo, hi = (theta, hi) if 3.8 * np.sin(theta / 2) / np.sin(theta / (2 * m)) > d else (lo, theta)
            R = 1.9 / np.sin(theta / (2 * m))
            e1 = (end - start) / d
            e2 = np.r_[0, 0, np.sign(start[2])] - np.dot(np.r_[0, 0, np.sign(start[2])], e1) * e1
            e2 /= np.linalg.norm(e2)
            c = (start + end) / 2 - R * np.cos(theta / 2) * e2
            for k in range(1, m):
                psi = -theta / 2 + k * theta / m
                pts.append(c + R * (np.sin(psi) * e1 + np.cos(psi) * e2))
            ss += "L" * loop_length
        pts.extend(helix)
        ss += "H" * helix_length
    coords = np.array(pts)
    if noise > 0:
        coords = coords + rng.normal(0, noise, coords.shape)
    D = np.linalg.norm(coords[:, None] - coords[None], axis=-1)
    sep = np.abs(np.arange(len(coords))[:, None] - np.arange(len(coords))[None])
    return dict(coords=coords, ss=ss, distances=D, contacts=(D < 8.0) & (sep >= 3))


def log_normalize(counts, target_sum: float = 1e4):
    """Library-size normalisation + log: log(1 + target_sum · count / total count of the cell)."""
    counts = np.asarray(counts, float)
    total = counts.sum(axis=1, keepdims=True)
    return np.log1p(target_sum * counts / np.where(total > 0, total, 1.0))


def pca(X, n_components: int = 2):
    """Principal component analysis by SVD of the column-centred data.

    Returns ``(scores, components, explained_variance_ratio)``: scores are N × k, components k × J.
    """
    X = np.asarray(X, float)
    Xc = X - X.mean(axis=0)
    U, S, Vt = np.linalg.svd(Xc, full_matrices=False)
    var = S ** 2 / (len(X) - 1)
    return U[:, :n_components] * S[:n_components], Vt[:n_components], var[:n_components] / var.sum()


def plot_embedding(Z, labels, ax=None, title: str = "", s: float = 4, legend: bool = True, cmap=None):
    """Scatter a 2-D embedding (PCA, t-SNE, latent space, …) coloured by a categorical or numeric label."""
    Z = np.asarray(Z, float)
    labels = np.asarray(labels)
    ax = ax or plt.gca()
    if labels.dtype.kind in "fc":
        sc = ax.scatter(Z[:, 0], Z[:, 1], c=labels, s=s, cmap=cmap or "viridis", linewidths=0)
        plt.colorbar(sc, ax=ax, shrink=0.8)
    else:
        palette = plt.get_cmap("tab10" if len(np.unique(labels)) <= 10 else "tab20")
        for k, lab in enumerate(np.unique(labels)):
            m = labels == lab
            ax.scatter(Z[m, 0], Z[m, 1], s=s, color=palette(k % palette.N), label=str(lab), linewidths=0)
        if legend:
            ax.legend(markerscale=3, fontsize=7, frameon=False, loc="best")
    ax.set_xticks([]); ax.set_yticks([])
    if title:
        ax.set_title(title)
    return ax


print("🧰 Toolkit loaded:", ", ".join(['sample_negative_binomial', 'make_expression_programs', 'simulate_scrna', 'simulate_helix_bundle', 'log_normalize', 'pca', 'plot_embedding']))

---
## 1 · The generative-modelling landscape

> 📍 **Where we are.** A generative model learns $p(x)$ from samples $x^{(1)},\dots,x^{(N)}$ so that it can produce new
> samples. You already know two: the Gaussian mixture of [Module 13](13_mixture_models_em.ipynb) (pick a component, then
> draw from its Gaussian) and the VAE of [Module 25](25_variational_inference_vae.ipynb) (draw $z\sim\mathcal N(0,I)$,
> then decode). This section sets up the toy data, the yardsticks we will use for *every* model, and the map of the field.

A generative model can offer three things, and the families differ in which they deliver:

| Family | Sample? | Evaluate $p(x)$? | Main idea | Biology example |
|---|---|---|---|---|
| Mixture models ([Module 13](13_mixture_models_em.ipynb)) | yes | yes | a few Gaussian blobs | cell populations in flow cytometry |
| Autoregressive models ([Module 26](26_attention_transformers.ipynb)) | yes, one token at a time | yes | $p(x)=\prod_i p(x_i\mid x_{<i})$ | protein language models (ProGen) |
| VAEs ([Module 25](25_variational_inference_vae.ipynb)) | yes | lower bound only | latent $z$ + decoder | scVI |
| GANs | yes | no | generator vs discriminator | early image and molecule generators |
| Normalising flows | yes | yes (exactly) | invertible maps of noise | density estimation, physics |
| **Diffusion / flow matching** | yes, in many steps | yes, with extra work | learn to undo noise / follow a velocity field | RFdiffusion, AlphaFold 3, cell generators |

Our toy data set is a **differentiation fork**: a stem of progenitor cells that splits into two fates, A and B, drawn
in a 2-D state space (think of the first two coordinates of an embedding of differentiating cells). Its hidden truth is
simple and fully known: the three curves, the noise width around them (0.12), and the shares of cells on the stem and
on the two fates (50 %, 30 % and 20 %).

In [ ]:
# ── 1.1 A SIMULATED differentiation fork with known structure ──
FORK_SHARES = np.array([0.5, 0.3, 0.2])        # stem (progenitors), fate A, fate B
FORK_WIDTH = 0.12                              # biological + technical scatter around each curve


def fork_curves(s):
    """Noise-free points of the stem, fate A and fate B at curve positions s ∈ [0, 1] (each an array s.size × 2)."""
    stem = np.c_[np.zeros_like(s), -2 + 2 * s]
    fate_a = np.c_[-1.6 * s, 1.8 * np.sqrt(s)]
    fate_b = np.c_[1.6 * s, 1.8 * np.sqrt(s)]
    return stem, fate_a, fate_b


def sample_fork(rng, n, shares=FORK_SHARES, width=FORK_WIDTH):
    """n cells of the fork: choose a branch, a position along it, add Gaussian scatter. Returns (X, branch 0/1/2)."""
    branch = rng.choice(3, n, p=shares)
    curves = np.stack(fork_curves(rng.random(n)))              # 3 × n × 2: every cell on every curve ...
    X = curves[branch, np.arange(n)] + rng.normal(0, width, (n, 2))   # ... keep the curve of its own branch
    return X, branch


rng = np.random.default_rng(SEED + 1)
X_fork, branch_fork = sample_fork(rng, 6000)
X_train, X_test = X_fork[:4000], X_fork[4000:]                # models see only the training cells
BRANCH_NAMES = ["stem", "fate A", "fate B"]
fig, ax = plt.subplots(figsize=(4.6, 4))
for k in range(3):
    m = branch_fork[:4000] == k
    ax.scatter(*X_train[m].T, s=3, color=OKABE_ITO[k], label=f"{BRANCH_NAMES[k]} ({100 * m.mean():.0f} %)")
for c in fork_curves(np.linspace(0, 1, 200)):
    ax.plot(*c.T, color="k", lw=0.8)
ax.set_xlabel("state coordinate 1"); ax.set_ylabel("state coordinate 2"); ax.set_aspect("equal")
ax.set_title("SIMULATED fork: 4,000 training cells (black = true curves)"); ax.legend(markerscale=4, loc="lower right")
plt.tight_layout(); plt.show()

**Yardsticks.** How do we know whether generated cells are good? Three complementary checks, all possible only because
we know the truth:

1. **Energy distance** between the generated sample $G$ and held-out real cells $R$:

$$\mathcal E(G,R)=2\,\mathbb E\lVert g-r\rVert-\mathbb E\lVert g-g'\rVert-\mathbb E\lVert r-r'\rVert ,$$

*In words:* the average distance *between* the two samples, minus the average distances *within* each sample. It is
zero when the two distributions are identical and positive otherwise (Székely & Rizzo 2013), so it compares whole
distributions without any density estimate. (Its cousin, the maximum mean discrepancy, MMD, uses a kernel instead of
the distance; Gretton *et al.* 2012.) Two finite samples from the *same* distribution still give a small positive
value — the **noise floor** we compare against.

2. **Off-fork fraction**: the share of generated cells farther than $3\times0.12$ from every true curve — cells that no
real cell resembles.
3. **Branch shares**: assign each on-fork cell to its nearest curve and compare the shares with the true 50 / 30 / 20 %.

In [ ]:
# ── 1.2 Yardsticks, and a Gaussian mixture (Module 13) used as a sampler ──
SKELETON = np.vstack(fork_curves(np.linspace(0, 1, 400)))          # 1,200 points on the true curves
SKELETON_BRANCH = np.repeat([0, 1, 2], 400)


def energy_distance(A, B):
    """2 E|a − b| − E|a − a'| − E|b − b'| for two samples (rows = points); 0 iff the distributions are equal."""
    return 2 * cdist(A, B).mean() - cdist(A, A).mean() - cdist(B, B).mean()


def fork_report(G, name):
    """One row of yardsticks for a generated sample G of the fork."""
    d = cdist(G, SKELETON)
    on_fork = d.min(1) < 3 * FORK_WIDTH
    shares = np.bincount(SKELETON_BRANCH[d.argmin(1)][on_fork], minlength=3) / max(on_fork.sum(), 1)
    return {"sample": name, "energy distance": energy_distance(G, X_test), "off-fork %": 100 * (1 - on_fork.mean()),
            "stem %": 100 * shares[0], "fate A %": 100 * shares[1], "fate B %": 100 * shares[2]}


rows = [fork_report(X_train[:2000], "2,000 real training cells (noise floor)")]
gmm_samples = {}
for K in (3, 12):
    gmm = GaussianMixture(K, random_state=SEED).fit(X_train)          # EM, exactly as in Module 13
    gmm_samples[K] = gmm.sample(2000)[0]
    rows.append(fork_report(gmm_samples[K], f"Gaussian mixture, K = {K}"))
report = pd.DataFrame(rows).set_index("sample")
print("truth: stem 50 %, fate A 30 %, fate B 20 %; every number compares 2,000 cells with 2,000 held-out real cells")
print(report.round(4).to_string())
fig, axes = plt.subplots(1, 2, figsize=(8.4, 3.6), sharex=True, sharey=True)
for ax, K in zip(axes, (3, 12)):
    ax.scatter(*X_test.T, s=2, color="0.8", label="held-out real cells")
    ax.scatter(*gmm_samples[K].T, s=2, color=OKABE_ITO[0], label=f"GMM samples (K = {K})")
    ax.set_title(f"Gaussian mixture with K = {K} components"); ax.set_aspect("equal"); ax.legend(markerscale=4, fontsize=7)
plt.tight_layout(); plt.show()

🔍 **What just happened?** The table's first row is the noise floor: two independent samples of 2,000 real cells differ
by an energy distance of about 0.001 and agree on the branch shares to within one percentage point. Three full-covariance
Gaussians — one elongated blob per branch — already do surprisingly well, because each branch of this fork is nearly
straight: the energy distance is about four times the floor and about 2 % of the samples fall off the fork, mostly in
Gaussian tails that overshoot the tips of the branches (left panel). Twelve components bring the off-fork fraction down to
0.25 %. **In two dimensions, a mixture with enough components is a strong baseline** — keep that in mind when the neural
models arrive. The trouble is scaling: the number of components needed grows quickly with curvature and thinness of the
structure and explodes with dimension (thousands of genes, thousands of atoms, and constraints such as 3.8 Å between
neighbouring Cα atoms that no blob can express). We need models that learn arbitrary shapes without being told their form.

**Two classic answers, in one paragraph each.**

* **GANs** (Goodfellow *et al.* 2014) train a *generator* network that maps noise to samples against a *discriminator*
  network that tries to tell generated from real samples. They produce sharp samples in one network evaluation, but
  training is a delicate two-player game, they give no density, and they are prone to **mode collapse** — covering only
  part of the data (e.g. generating fate A cells and never fate B).
* **Normalising flows** push simple noise through an *invertible* map $x=f(z)$ and compute the exact density of $x$ with
  the change-of-variables formula below. Their constraint — every layer must be invertible with a cheap Jacobian
  determinant, as in the coupling layers of Real NVP (Dinh, Sohl-Dickstein & Bengio 2017) — limits the architectures.

🧮 **Math refresher — the change-of-variables formula.** If $z$ has density $p_Z$ and $x=f(z)$ with $f$ invertible and
increasing, then probability is conserved in every small interval: $p_X(x)\,dx=p_Z(z)\,dz$, so

$$p_X(x)=p_Z\big(f^{-1}(x)\big)\left|\frac{d f^{-1}(x)}{dx}\right| \qquad\Big(\text{in } d \text{ dimensions: }\big|\det J_{f^{-1}}(x)\big|\Big).$$

*In words:* the density of the transformed variable is the density of the original, times how much the map squeezes
(density goes up) or stretches (density goes down) space at that point. The Jacobian determinant measures that local
change of volume.

In [ ]:
# ── 1.3 Change of variables in 1-D: log-normal gene expression from Gaussian noise ──
rng = np.random.default_rng(SEED + 13)
mu_log, sd_log = 1.0, 0.6                          # a gene whose log-expression is N(1, 0.6²)
z = rng.standard_normal(100_000)                   # simple noise ...
x_expr = np.exp(mu_log + sd_log * z)               # ... pushed through the increasing map f(z) = exp(μ + σ z)
x_grid = np.linspace(0.05, 12, 400)
z_back = (np.log(x_grid) - mu_log) / sd_log        # f⁻¹(x)
dens = np.exp(-z_back ** 2 / 2) / np.sqrt(2 * np.pi) * (1 / (sd_log * x_grid))   # p_Z(f⁻¹(x)) · |d f⁻¹/dx|
fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 3.2))
a1.hist(z, bins=100, density=True, color=OKABE_ITO[5]); a1.set_title("noise z ~ N(0, 1)"); a1.set_xlabel("z")
a2.hist(x_expr, bins=np.linspace(0, 12, 120), density=True, color="0.75", label="samples of x = exp(1 + 0.6 z)")
a2.plot(x_grid, dens, color=OKABE_ITO[1], lw=2, label="change-of-variables formula")
a2.set_xlabel("expression x"); a2.set_title("the transformed density is exactly predicted"); a2.legend(fontsize=7)
plt.tight_layout(); plt.show()
z_at_2 = (np.log(2) - mu_log) / sd_log                   # P(x < 2) = P(z < f⁻¹(2)): the formula, integrated exactly
print(f"fraction of samples below x = 2: empirical {np.mean(x_expr < 2):.4f} | "
      f"from the formula {0.5 * (1 + math.erf(z_at_2 / np.sqrt(2))):.4f}")

🔍 **What just happened?** A straight Gaussian became a skewed, log-normal expression distribution, and the formula
predicts the histogram exactly (the two fractions agree to within 0.001, the sampling noise of 100,000 draws). A normalising flow
does the same with a *learned*, multi-layer invertible map in many dimensions. Keep this picture: **noise in, data out,
through a transformation.** Diffusion and flow matching keep the idea but build the transformation out of many small,
learned steps, each of which only has to be approximately right — no invertibility constraint on the network.

✏️ **Your turn.** In cell 1.2, fit mixtures with $K=6$ and $K=30$. How do the energy distance and the off-fork fraction
change? At what point does adding components stop helping, and what would happen in 2,000 dimensions?

---
## 2 · Scores and Langevin dynamics

> 📍 **Where we are.** Before learning anything, we ask: *if we knew the shape of $p(x)$, how would we sample from it?*
> The answer — follow the gradient of $\log p$, plus noise — is the engine of every diffusion model.

The **score** of a distribution is the gradient of its log-density with respect to the data point:

$$s(x)=\nabla_x\log p(x).$$

*In words:* at every point the score is an arrow pointing uphill towards more probable data, longer where the
probability rises steeply. Crucially, if $p(x)=\tilde p(x)/Z$ with an unknown normalising constant $Z$, then
$\nabla\log p=\nabla\log\tilde p$: **the score never needs the normalising constant** — the same reason
Metropolis–Hastings ([Module 21](21_bayesian_learning_mcmc.ipynb)) needs only ratios of probabilities.

**Langevin dynamics** turns a score into a sampler. Start anywhere and repeat

$$x_{k+1}=x_k+\frac{\eta}{2}\,\nabla_x\log p(x_k)+\sqrt{\eta}\;\varepsilon_k,\qquad \varepsilon_k\sim\mathcal N(0,I).$$

*In words:* take a small step uphill and add a little random noise. The uphill step pulls the point towards probable
regions; the noise keeps it from collapsing onto the peak, so that after many steps (and for small $\eta$) the points are
distributed according to $p$ itself.

> 🔗 **Builds on [Module 21 · MCMC](21_bayesian_learning_mcmc.ipynb), Section 8.** Random-walk Metropolis–Hastings
> proposes blind random moves and accepts or rejects them. Langevin dynamics is a random walk *steered by the gradient*,
> so it wastes far fewer moves; adding Module 21's accept/reject step to it gives the Metropolis-adjusted Langevin
> algorithm (MALA; Roberts & Tweedie 1996), which removes the small bias of a finite step size $\eta$.

For a Gaussian mixture the score has a closed form — and it contains an old friend. With components
$\mathcal N(\mu_k,\sigma^2I)$ and weights $\pi_k$,

$$\nabla_x\log p(x)=\sum_k \gamma_k(x)\,\frac{\mu_k-x}{\sigma^2},\qquad \gamma_k(x)=\frac{\pi_k\,\mathcal N(x\mid\mu_k,\sigma^2I)}{\sum_j\pi_j\,\mathcal N(x\mid\mu_j,\sigma^2I)}.$$

*In words:* each component pulls the point towards its own mean, and the pulls are weighted by the
**responsibilities** $\gamma_k(x)$ — exactly the E-step of [Module 13](13_mixture_models_em.ipynb).

Our test landscape has two cell states far apart: a common state (80 % of cells) and a rare one (20 %).

In [ ]:
# ── 2.1 A two-state landscape with an exact score ──
W2, MU2, VAR2 = np.array([0.8, 0.2]), np.array([[-3.0, 0.0], [3.0, 0.0]]), 0.25   # weights, means, variance 0.5²


def mixture_log_density_and_score(X, weights, means, var):
    """log p(x) and the score ∇ log p(x) of an isotropic Gaussian mixture, for every row of X."""
    diff = X[:, None, :] - means[None]                                    # n × K × 2
    log_joint = np.log(weights) - 0.5 * (diff ** 2).sum(-1) / var - np.log(2 * np.pi * var)
    log_p = lse(log_joint, axis=1)
    resp = np.exp(log_joint - log_p[:, None])                             # responsibilities (Module 13's E-step)
    return log_p, -(resp[:, :, None] * diff).sum(1) / var


gx, gy = np.meshgrid(np.linspace(-6, 6, 200), np.linspace(-3, 3, 100))
grid = np.c_[gx.ravel(), gy.ravel()]
logp_grid, _ = mixture_log_density_and_score(grid, W2, MU2, VAR2)
qx, qy = np.meshgrid(np.linspace(-5.5, 5.5, 23), np.linspace(-2.5, 2.5, 11))
_, sc = mixture_log_density_and_score(np.c_[qx.ravel(), qy.ravel()], W2, MU2, VAR2)
fig, ax = plt.subplots(figsize=(9, 3.2))
ax.contourf(gx, gy, np.exp(logp_grid).reshape(gx.shape), levels=12, cmap="Greys")
length = np.linalg.norm(sc, axis=1, keepdims=True)
ax.quiver(qx.ravel(), qy.ravel(), *(sc / length).T, length.ravel(), cmap="viridis", scale=30, width=0.004)
ax.set_aspect("equal"); ax.set_xlabel("state coordinate 1"); ax.set_ylabel("state coordinate 2")
ax.set_title("SIMULATED two-state landscape: density (grey) and score direction (arrows; colour = length)")
plt.tight_layout(); plt.show()

In [ ]:
# ── 2.2 Langevin sampling with the exact score — and a 🧪 ground-truth check of the state shares ──
def langevin(X, score_fn, step, n_steps, rng):
    """Unadjusted Langevin dynamics: x ← x + (step/2)·score(x) + sqrt(step)·noise, for every row of X at once."""
    for _ in range(n_steps):
        X = X + 0.5 * step * score_fn(X) + np.sqrt(step) * rng.standard_normal(X.shape)
    return X


rng = np.random.default_rng(SEED + 2)
start = rng.uniform([-6, -3], [6, 3], (2000, 2))                       # 2,000 chains started anywhere
exact_score = lambda X: mixture_log_density_and_score(X, W2, MU2, VAR2)[1]
plain = langevin(start, exact_score, step=0.01, n_steps=1000, rng=rng)

annealed = start.copy()                                                # same starts, but a sequence of noise levels
NOISE_LEVELS = np.geomspace(4.0, 0.05, 10)
for sigma in NOISE_LEVELS:                                             # p_σ = p convolved with N(0, σ²I): a mixture too
    smoothed_score = lambda X, s=sigma: mixture_log_density_and_score(X, W2, MU2, VAR2 + s ** 2)[1]
    annealed = langevin(annealed, smoothed_score, step=0.1 * (VAR2 + sigma ** 2), n_steps=100, rng=rng)
for name, S in [("plain Langevin", plain), ("annealed Langevin", annealed)]:
    left = S[:, 0] < 0
    print(f"{name:18s}: common state {100 * left.mean():.1f} %, rare state {100 * (1 - left.mean()):.1f} %  "
          f"(truth 80 / 20);  spread within the common state sd = {S[left].std(0).mean():.2f} (truth 0.50)")
print(f"chains that started on the left half: {100 * np.mean(start[:, 0] < 0):.1f} %")

In [ ]:
# ── 2.3 Picture: where the chains end ──
fig, axes = plt.subplots(1, 3, figsize=(12, 3.2), gridspec_kw={"width_ratios": [2, 2, 1.2]})
for ax, (name, S) in zip(axes, [("plain Langevin (one noise level)", plain), ("annealed Langevin (σ = 4 → 0.05)", annealed)]):
    ax.contour(gx, gy, np.exp(logp_grid).reshape(gx.shape), levels=6, colors="0.6", linewidths=0.6)
    ax.scatter(*S.T, s=2, color=OKABE_ITO[0]); ax.set_xlim(-6, 6); ax.set_ylim(-3, 3); ax.set_aspect("equal")
    ax.set_title(name)
shares = [[80, 20], [100 * np.mean(plain[:, 0] < 0), 100 * np.mean(plain[:, 0] >= 0)],
          [100 * np.mean(annealed[:, 0] < 0), 100 * np.mean(annealed[:, 0] >= 0)]]
for k, (lab, col) in enumerate([("truth", "0.3"), ("plain", OKABE_ITO[1]), ("annealed", OKABE_ITO[2])]):
    axes[2].bar(np.arange(2) + (k - 1) * 0.27, shares[k], width=0.27, color=col, label=lab)
axes[2].set_xticks([0, 1], ["common", "rare"]); axes[2].set_ylabel("% of samples"); axes[2].legend(fontsize=7)
axes[2].set_title("state shares")
plt.tight_layout(); plt.show()

🔍 **What just happened?** Both samplers put their points in the right places with the right spread, but plain
Langevin gets the **proportions** wrong: it reports about 50 / 50 instead of 80 / 20. The reason is visible in the score
field of 2.1: far from both states the arrows point to whichever state is *nearer*, and the valley between the states
is a region of almost zero probability that a chain with small steps practically never crosses. Each chain simply stays
in the basin where it started, so the proportions reflect the starting points (half of them on the left), not the
probabilities. The *weights* of the states are invisible to a local sampler.

**Annealing over noise levels fixes this.** Blurring $p$ with Gaussian noise of standard deviation $\sigma$ gives a
smoothed density $p_\sigma$; for $\sigma=4$ the two states merge into one broad hill whose slopes "know" that the left
state carries more mass, so chains drift there in the right proportions. Lowering $\sigma$ step by step then sharpens
the hill back into the two states while the chains stay put. The annealed sampler recovers shares close to 80 / 20
(about 77 / 23; the small remaining bias comes from the finite number of steps spent at the noise levels where the two
states separate).

This was the key insight of **noise-conditional score networks** (Song & Ermon 2019): learn the score of $p_\sigma$ for
*a whole range* of noise levels, then sample by annealed Langevin dynamics. Learning helps for a second reason too: a
score estimated from data is unreliable exactly where there are no data — in the valleys — but the noise-blurred
densities have data everywhere. Diffusion models (next sections) are this idea with a particular, carefully designed
sequence of noise levels.

⚠️ **Pitfall.** Langevin dynamics with a finite step is slightly biased (the spread within a state comes out a little
larger than the true 0.50), and with too large a step it diverges. Production samplers use step-size schedules,
corrector steps or the accept/reject step of MALA.

✏️ **Your turn.** Rerun 2.2 with the two states at $\pm1.5$ instead of $\pm3$. Does plain Langevin still get the
proportions wrong? (Hint: when the valley is shallow, chains cross it.)

---
## 3 · Forward diffusion: turning data into noise

> 📍 **Where we are.** We need scores of noise-blurred versions of the data distribution at many noise levels. The
> **denoising diffusion probabilistic model** (DDPM; Ho, Jain & Abbeel 2020, building on Sohl-Dickstein *et al.* 2015)
> defines those levels through a simple Markov chain that destroys the data a little at a time.

**The forward chain.** Choose a *noise schedule* $\beta_1,\dots,\beta_T$ (small numbers, e.g. from $10^{-4}$ to $0.02$
over $T=1000$ steps) and let

$$q(x_t\mid x_{t-1})=\mathcal N\!\big(\sqrt{1-\beta_t}\;x_{t-1},\;\beta_t I\big),\qquad\text{i.e. } x_t=\sqrt{\alpha_t}\,x_{t-1}+\sqrt{\beta_t}\,\varepsilon_t,\quad \alpha_t=1-\beta_t .$$

*In words:* at each step shrink the point slightly towards the origin and add a little fresh Gaussian noise. The
shrinking is chosen so that the total variance stays constant: if $x_{t-1}$ has variance 1, so does $x_t$
($\alpha_t\cdot1+\beta_t=1$).

**Jumping to any step in one go.** Two steps from $x_0$:

$$x_2=\sqrt{\alpha_2}\big(\sqrt{\alpha_1}x_0+\sqrt{\beta_1}\varepsilon_1\big)+\sqrt{\beta_2}\varepsilon_2
=\sqrt{\alpha_1\alpha_2}\,x_0+\underbrace{\sqrt{\alpha_2\beta_1}\,\varepsilon_1+\sqrt{\beta_2}\,\varepsilon_2}_{\text{Gaussian, variance }\alpha_2(1-\alpha_1)+(1-\alpha_2)\,=\,1-\alpha_1\alpha_2}.$$

A sum of independent Gaussians is Gaussian and the variances add, so by induction, with $\bar\alpha_t=\prod_{s=1}^t\alpha_s$,

$$q(x_t\mid x_0)=\mathcal N\!\big(\sqrt{\bar\alpha_t}\,x_0,\;(1-\bar\alpha_t)\,I\big),\qquad x_t=\sqrt{\bar\alpha_t}\,x_0+\sqrt{1-\bar\alpha_t}\;\varepsilon .$$

*In words:* the noisy version at step $t$ is a shrunken copy of the data plus Gaussian noise; $\bar\alpha_t$ falls from
1 (clean data) to almost 0 (pure noise). We never need to run the chain: one line produces $x_t$ for any $t$. As
$t\to T$ the data are forgotten and $x_T\approx\mathcal N(0,I)$ — a distribution we can sample trivially.

> 🔗 **Builds on [Module 21](21_bayesian_learning_mcmc.ipynb), Section 7.** This is a Markov chain whose stationary
> distribution is $\mathcal N(0,I)$: run it long enough and the start is forgotten — just as an MCMC chain forgets its
> starting point after burn-in. Generation will run a learned chain the other way: from the stationary distribution
> back to the data.

In [ ]:
# ── 3.1 Two noise schedules: linear (Ho et al. 2020) and cosine (Nichol & Dhariwal 2021) ──
T_STEPS = 1000
BETAS = np.linspace(1e-4, 0.02, T_STEPS)                    # β_1 … β_T (index t − 1)
ABAR = np.cumprod(1 - BETAS)                                 # ᾱ_t
f_cos = np.cos((np.arange(T_STEPS + 1) / T_STEPS + 0.008) / 1.008 * np.pi / 2) ** 2
ABAR_COS = f_cos[1:] / f_cos[0]
t_axis = np.arange(1, T_STEPS + 1)
fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 3.2))
for ab, lab, col in [(ABAR, "linear β", OKABE_ITO[0]), (ABAR_COS, "cosine", OKABE_ITO[1])]:
    a1.plot(t_axis, ab, color=col, label=lab)
    a2.plot(t_axis, np.log10(ab / (1 - ab)), color=col, label=lab)
a1.set_xlabel("step t"); a1.set_ylabel(r"$\bar\alpha_t$ (signal kept)"); a1.set_title("how much of the data survives")
a2.set_xlabel("step t"); a2.set_ylabel(r"log$_{10}$ SNR = log$_{10}\,\bar\alpha_t/(1-\bar\alpha_t)$")
a2.set_title("signal-to-noise ratio"); a1.legend(); a2.axhline(0, color="k", lw=0.5); a2.set_ylim(-6, 5)
plt.tight_layout(); plt.show()
print(f"linear schedule: ᾱ at t = 100, 500, 1000: {ABAR[99]:.3f}, {ABAR[499]:.4f}, {ABAR[-1]:.1e}")
print(f"cosine schedule: ᾱ at t = 100, 500, 1000: {ABAR_COS[99]:.3f}, {ABAR_COS[499]:.4f}, {ABAR_COS[-1]:.1e}")

In [ ]:
# ── 3.2 🧪 The closed form agrees with running the chain step by step; the fork dissolves ──
rng = np.random.default_rng(SEED + 3)
x_chain = X_train.copy()
snapshots = {0: X_train.copy()}
for t in range(1, T_STEPS + 1):                                # 1,000 small steps of the Markov chain
    x_chain = np.sqrt(1 - BETAS[t - 1]) * x_chain + np.sqrt(BETAS[t - 1]) * rng.standard_normal(x_chain.shape)
    if t in (50, 150, 300, 600, 1000):
        snapshots[t] = x_chain.copy()
print(" step t | mean (chain)    mean (formula) | total variance: chain  formula | energy distance chain vs one jump")
for t in (50, 150, 300, 600, 1000):
    jump = np.sqrt(ABAR[t - 1]) * X_train + np.sqrt(1 - ABAR[t - 1]) * rng.standard_normal(X_train.shape)
    m_formula = np.sqrt(ABAR[t - 1]) * X_train.mean(0)
    v_formula = ABAR[t - 1] * X_train.var(0).sum() + 2 * (1 - ABAR[t - 1])
    print(f"  {t:5d} | {np.round(snapshots[t].mean(0), 2)}   {np.round(m_formula, 2)}   |  {snapshots[t].var(0).sum():.3f}"
          f"   {v_formula:.3f}   | {energy_distance(snapshots[t][:2000], jump[:2000]):.4f}")
fig, axes = plt.subplots(1, 6, figsize=(12, 2.4), sharex=True, sharey=True)
for ax, (t, S) in zip(axes, snapshots.items()):
    ax.scatter(*S[:1500].T, s=1, c=branch_fork[:1500], cmap=plt.matplotlib.colors.ListedColormap(OKABE_ITO[:3]))
    ax.set_title(f"t = {t}" + ("" if t == 0 else f"  (ᾱ = {ABAR[t - 1]:.2f})"), fontsize=8)
    ax.set_xlim(-3.5, 3.5); ax.set_ylim(-3.5, 3.5); ax.set_aspect("equal"); ax.set_xticks([]); ax.set_yticks([])
fig.suptitle("Forward diffusion of the SIMULATED fork (colours = true branch)", fontsize=10)
plt.tight_layout(); plt.show()

🔍 **What just happened?** Running 1,000 small noisy steps and taking a single jump with $q(x_t\mid x_0)$ give the same
means and variances, and their energy distance is at the noise floor at every step: the derivation holds. The pictures
show the fork dissolving: by $t=150$ the branches are blurred, by $t=300$ only a vague blob remains, and at $t=1000$ the
cloud is a standard Gaussian (total variance 2 = one per dimension) with no memory of the data.

The two schedules differ in *where* they spend their steps: the linear schedule destroys the signal quickly (the SNR
crosses 1 already around $t\approx 250$, the cosine schedule only around $t\approx 500$), so the cosine schedule keeps a
useful signal for longer and spends more
steps at intermediate noise, which Nichol & Dhariwal (2021) found improves sample quality. The choice of schedule is one
of the main design decisions of a diffusion model (Karras *et al.* 2022 analyse it systematically).

✏️ **Your turn.** Repeat the dissolving pictures with the cosine schedule (`ABAR_COS`). At which step is the fork still
recognisable under each schedule?

---
## 4 · Learning to denoise (DDPM)

> 📍 **Where we are.** Destroying data is easy; generating requires running the chain *backwards*:
> $x_T\sim\mathcal N(0,I)\to x_{T-1}\to\dots\to x_0$. Each backward step needs to know, roughly, which direction leads
> back to the data. That is what a neural network will learn.

**The reverse step.** For small $\beta_t$ the reverse step $p(x_{t-1}\mid x_t)$ is approximately Gaussian. If we knew the
clean point $x_0$, the reverse step would even be exact: Bayes' rule on Gaussians gives
$q(x_{t-1}\mid x_t,x_0)=\mathcal N(\tilde\mu_t,\tilde\beta_tI)$ with

$$\tilde\mu_t=\frac{1}{\sqrt{\alpha_t}}\Big(x_t-\frac{\beta_t}{\sqrt{1-\bar\alpha_t}}\,\varepsilon\Big),\qquad
\tilde\beta_t=\frac{1-\bar\alpha_{t-1}}{1-\bar\alpha_t}\,\beta_t ,$$

where $\varepsilon$ is the noise that turned $x_0$ into $x_t$. *In words:* to step back, remove a small multiple of the
noise that is in $x_t$, rescale, and add a little fresh noise of variance $\tilde\beta_t$.

**The network.** We do not know $\varepsilon$ when generating, so we train a network $\varepsilon_\theta(x_t,t)$ to
predict it, by minimising

$$\mathcal L(\theta)=\mathbb E_{x_0,\;t,\;\varepsilon}\Big[\big\lVert\varepsilon-\varepsilon_\theta\big(\sqrt{\bar\alpha_t}\,x_0+\sqrt{1-\bar\alpha_t}\,\varepsilon,\;t\big)\big\rVert^2\Big].$$

*In words:* take a training point, pick a random step $t$, add the corresponding amount of noise in one jump, and ask
the network which noise was added. It is ordinary regression with a squared loss
([Module 22](22_neural_networks.ipynb)) on an endless supply of freshly noised examples. Ho *et al.* (2020) derived it
as a simplified form of a variational bound (the ELBO of [Module 25](25_variational_inference_vae.ipynb)): a diffusion
model is a hierarchical VAE whose encoder is fixed (the noising chain) and whose decoder is the learned reverse chain.

**Why predict the noise?** The target $\varepsilon$ always has variance 1 whatever $t$ is, which makes the regression
well scaled at every noise level; predicting $x_0$ directly is equivalent mathematically but weights the noise levels
differently, and Ho *et al.* found that ε-prediction trained better.

**The connection to scores (denoising score matching).** The noised density has score
$\nabla_{x_t}\log q(x_t\mid x_0)=-\varepsilon/\sqrt{1-\bar\alpha_t}$, and the best possible regression predicts the
*average* noise given $x_t$. It follows (Vincent 2011) that

$$\nabla_{x}\log q_t(x)\;\approx\;-\frac{\varepsilon_\theta(x,t)}{\sqrt{1-\bar\alpha_t}} .$$

*In words:* **a trained denoiser is a score estimator** for every noise level at once — exactly what Section 2 said we
need. "Predict the noise" and "point uphill towards the data" are the same instruction.

🧮 **Math refresher — telling the network what time it is (Fourier features).** The network must behave differently at
$t=10$ (remove a whisper of noise) and $t=900$ (invent structure from scratch). Feeding $t$ as a single number makes it
hard to learn sharp changes, so we feed $\sin(\omega_k t)$ and $\cos(\omega_k t)$ for a range of frequencies
$\omega_k$ — the same trick as the positional encodings of transformers ([Module 26](26_attention_transformers.ipynb)):
low frequencies tell coarse time, high frequencies fine time.

In [ ]:
# ── 4.1 The network: an MLP that sees the noisy point, Fourier features of t and (optionally) a label ──
class FourierMLP(nn.Module):
    """f_θ(x, t[, label]) for x ∈ R^dim and t ∈ [0, 1]; used as a noise predictor (DDPM) or a velocity field (flow matching)."""

    def __init__(self, dim=2, hidden=128, n_freq=8, n_labels=0, label_dim=16):
        super().__init__()
        self.dim, self.n_labels = dim, n_labels
        self.register_buffer("freqs", torch.exp(torch.linspace(0.0, np.log(500.0), n_freq)))   # ω from 1 to 500
        self.label_emb = nn.Embedding(n_labels + 1, label_dim) if n_labels else None   # index n_labels = "no label"
        d_in = dim + 2 * n_freq + (label_dim if n_labels else 0)
        self.net = nn.Sequential(nn.Linear(d_in, hidden), nn.SiLU(), nn.Linear(hidden, hidden), nn.SiLU(),
                                 nn.Linear(hidden, hidden), nn.SiLU(), nn.Linear(hidden, dim))

    def forward(self, x, t, label=None):
        angles = t[:, None] * self.freqs[None]
        parts = [x, torch.sin(angles), torch.cos(angles)]
        if self.label_emb is not None:
            parts.append(self.label_emb(label))
        return self.net(torch.cat(parts, dim=1))


ABAR_T = torch.tensor(ABAR, dtype=torch.float32)


def train_ddpm(X, n_steps=5000, batch=256, lr=1e-3, hidden=128, seed=SEED):
    """Train an ε-prediction network on the rows of X with the DDPM loss; returns (model, loss per step)."""
    torch.manual_seed(seed)
    model = FourierMLP(dim=X.shape[1], hidden=hidden)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, n_steps)    # learning rate decays smoothly to 0
    data, losses = torch.tensor(X, dtype=torch.float32), []
    for _ in range(n_steps):
        x0 = data[torch.randint(0, len(data), (batch,))]
        t = torch.randint(1, T_STEPS + 1, (batch,))                    # a random step for every example
        eps = torch.randn_like(x0)
        ab = ABAR_T[t - 1][:, None]
        x_t = ab.sqrt() * x0 + (1 - ab).sqrt() * eps                   # one jump with q(x_t | x_0)
        loss = ((model(x_t, t / T_STEPS) - eps) ** 2).mean()          # "which noise was added?"
        opt.zero_grad(); loss.backward(); opt.step(); sched.step()
        losses.append(loss.item())
    return model, np.array(losses)


t0 = time.time()
ddpm_fork, ddpm_losses = train_ddpm(X_train)
print(f"DDPM trained on {len(X_train)} cells: 5,000 Adam steps in {time.time() - t0:.1f} s; "
      f"loss {ddpm_losses[:100].mean():.3f} (first 100 steps) → {ddpm_losses[-300:].mean():.3f} (last 300)")

In [ ]:
# ── 4.2 Ancestral sampling: run the learned chain backwards from pure noise ──
@torch.no_grad()
def ddpm_sample(model, n, n_steps=T_STEPS, seed=SEED, return_path=False):
    """DDPM ancestral sampling over n_steps evenly spaced steps (n_steps = T is the original chain; fewer steps = longer jumps)."""
    g = torch.Generator().manual_seed(seed)
    steps = np.unique(np.linspace(1, T_STEPS, n_steps).round().astype(int))[::-1]   # T, …, 1
    x = torch.randn(n, model.dim, generator=g)
    path = [x.numpy().copy()]
    for k, t in enumerate(steps):
        ab_t = float(ABAR[t - 1])
        ab_prev = float(ABAR[steps[k + 1] - 1]) if k + 1 < len(steps) else 1.0
        beta = 1 - ab_t / ab_prev                                     # the β of this (possibly long) jump
        eps = model(x, torch.full((n,), t / T_STEPS))
        x = (x - beta / np.sqrt(1 - ab_t) * eps) / np.sqrt(1 - beta)  # mean of the reverse step
        if k + 1 < len(steps):                                        # add fresh noise, except at the very end
            x = x + np.sqrt(beta * (1 - ab_prev) / (1 - ab_t)) * torch.randn(n, model.dim, generator=g)
        path.append(x.numpy().copy())
    return (x.numpy(), np.array(path)) if return_path else x.numpy()


t0 = time.time()
ddpm_samples = ddpm_sample(ddpm_fork, 2000)
print(f"2,000 samples × 1,000 reverse steps in {time.time() - t0:.1f} s")
rows.append(fork_report(ddpm_samples, "DDPM, 1,000 steps"))
report = pd.DataFrame(rows).set_index("sample")
print(report.round(4).to_string())
fig, axes = plt.subplots(1, 3, figsize=(11, 3.8))
for ax, (S, title) in zip(axes, [(X_test, "held-out real cells"), (gmm_samples[12], "Gaussian mixture, K = 12"),
                                 (ddpm_samples, "DDPM samples (1,000 steps)")]):
    ax.scatter(*S[:2000].T, s=2, color=OKABE_ITO[0] if "DDPM" in title else "0.4")
    ax.set_title(title); ax.set_xlim(-2.4, 2.4); ax.set_ylim(-2.8, 2.4); ax.set_aspect("equal")
plt.tight_layout(); plt.show()

🔍 **What just happened?** A three-layer network with 128 units per layer, trained for about ten seconds, turned Gaussian
noise into cells that lie on the fork. Its energy distance (0.0018) is close to the noise floor (0.0011) and as good as
that of the 12-component mixture; 0.65 % of its samples fall off the fork (the mixture: 0.25 %), and its branch shares are
within about three percentage points of 50 / 30 / 20 (it makes slightly too many fate A cells at the expense of the
stem). A network that was told nothing about blobs or curves learned *where* the cells are and *how many* each fate
receives about as well as a mixture whose shape happens to suit this toy almost perfectly. The price is speed: one sample
needs 1,000 network evaluations instead of one.

Now two ground-truth checks that only a simulation allows. First, the claim that the denoiser *is* a score estimator.
For the two-state mixture of Section 2 the noised density $q_t$ is again a mixture (means $\sqrt{\bar\alpha_t}\mu_k$,
variance $\bar\alpha_t\sigma^2+1-\bar\alpha_t$), so its exact score is known at every $t$.

In [ ]:
# ── 4.3 🧪 Ground-truth check: the trained denoiser estimates the exact score of the noised data ──
rng = np.random.default_rng(SEED + 4)
comp = rng.random(4000) < W2[1]
X_two = MU2[comp.astype(int)] + rng.normal(0, np.sqrt(VAR2), (4000, 2))     # SIMULATED two-state cells
ddpm_two, _ = train_ddpm(X_two, n_steps=2000, hidden=64)
print("  step t | ᾱ_t   | cosine(learned, exact score) | relative error ‖learned − exact‖ / ‖exact‖")
for t in (20, 100, 300, 600):
    ab = ABAR[t - 1]
    x_t = np.sqrt(ab) * X_two[:1000] + np.sqrt(1 - ab) * rng.standard_normal((1000, 2))   # points where q_t lives
    _, exact = mixture_log_density_and_score(x_t, W2, np.sqrt(ab) * MU2, ab * VAR2 + 1 - ab)
    with torch.no_grad():
        eps_hat = ddpm_two(torch.tensor(x_t, dtype=torch.float32), torch.full((1000,), t / T_STEPS)).numpy()
    learned = -eps_hat / np.sqrt(1 - ab)                                   # score = −ε_θ / sqrt(1 − ᾱ_t)
    cos = np.sum(learned * exact, 1) / (np.linalg.norm(learned, axis=1) * np.linalg.norm(exact, axis=1))
    rel = np.linalg.norm(learned - exact, axis=1) / np.linalg.norm(exact, axis=1)
    print(f"  {t:6d} | {ab:.3f} |            {np.median(cos):.3f}            |   {np.median(rel):.3f}  (medians over 1,000 points)")

🔍 **What just happened?** The network was never shown a score; it only learned to guess added noise. Yet
$-\varepsilon_\theta/\sqrt{1-\bar\alpha_t}$ points in essentially the same direction as the exact score at every
noise level (median cosine ≥ 0.997), and for $t\ge100$ its length is right to within 5–11 %. Denoising *is* score
estimation. The exception is the smallest noise level, $t=20$: the direction is still right but the length is off by more
than 100 %. There the score is $-\varepsilon/\sqrt{1-\bar\alpha_t}$ with $\sqrt{1-\bar\alpha_t}\approx0.08$, so tiny errors
in the predicted noise are multiplied by about 13 — a known weak spot of ε-prediction at the lowest noise levels. It
matters little for sampling (the last steps only polish a sample that is already in place), and other parameterisations
handle it differently (Karras *et al.* 2022 discuss these choices).

The second check concerns a worry specific to generative models: has the model simply **memorised** its training cells?
If it had, its samples would sit much closer to training points than new real cells do.

In [ ]:
# ── 4.4 🧪 Memorisation check: distance to the nearest training cell ──
nn_generated = cdist(ddpm_samples, X_train).min(1)          # each generated cell → nearest training cell
nn_test = cdist(X_test, X_train).min(1)                     # each held-out real cell → nearest training cell
print(f"median distance to the nearest training cell: generated {np.median(nn_generated):.4f} | "
      f"held-out real {np.median(nn_test):.4f}  (ratio {np.median(nn_generated) / np.median(nn_test):.2f})")
print(f"generated cells closer than 0.002 to a training cell: {100 * np.mean(nn_generated < 0.002):.1f} % "
      f"(held-out real cells: {100 * np.mean(nn_test < 0.002):.1f} %)")
fig, ax = plt.subplots(figsize=(6.4, 3))
bins = np.linspace(0, 0.08, 60)
ax.hist(nn_test, bins=bins, alpha=0.6, density=True, color="0.5", label="held-out real cells")
ax.hist(nn_generated, bins=bins, alpha=0.6, density=True, color=OKABE_ITO[0], label="DDPM samples")
ax.set_xlabel("distance to the nearest training cell"); ax.set_ylabel("density"); ax.legend()
ax.set_title("A memorising model would pile up at 0; ours looks like new real data")
plt.tight_layout(); plt.show()

🔍 **What just happened?** Generated cells are about as far from their nearest training cell as genuinely new cells are
(the ratio of medians is 0.99, and about 2 % of both groups happen to fall within 0.002 of a training cell), so this
model generalises rather than memorises. Section 8 shows the opposite case
deliberately — and why "a perfect fit" would be a perfect memoriser.

⚠️ **Pitfall.** These checks are easy in 2-D with a known truth. In 20,000-dimensional expression space or in protein
structure space, nearest-neighbour distances need a meaningful metric (e.g. TM-score for structures,
[Module 38](38_protein_structure.ipynb)), and "close to a training example" may be legitimate (a natural protein family)
or a privacy problem (a patient's profile).

✏️ **Your turn.** Retrain the DDPM with `n_steps=300` (a tenth of the training). How do the energy distance, the
off-fork fraction and the branch shares in the table change? Which goes wrong first?

### 📄 Research connection — Origins of diffusion and score-based models

- **What the field does.** Sohl-Dickstein *et al.* (2015) proposed learning to reverse a gradual noising process, borrowing ideas from
    non-equilibrium thermodynamics. Song & Ermon (2019) trained a network to estimate the score at many noise levels
    (denoising score matching; Vincent 2011) and sampled with annealed Langevin dynamics. Ho, Jain & Abbeel (2020)
    reduced the training objective to ε-prediction (DDPM) and obtained image quality that rivalled GANs. Song *et al.*
    (2021) unified both views as stochastic differential equations and introduced the *probability-flow ODE*, a
    deterministic sampler that also gives exact likelihoods. Later work improved schedules and variances (Nichol &
    Dhariwal 2021), deterministic few-step sampling (DDIM; Song, Meng & Ermon 2021) and the overall design space
    (Karras *et al.* 2022).
- **What this notebook builds.** A DDPM with the original linear schedule and $T=1000$, an ε-prediction MLP with Fourier time features,
    ancestral sampling (with respaced steps in Section 5), and three checks against simulated truth: the learned score
    against an analytic score, sample quality (energy distance, off-fork fraction, branch shares) and a
    nearest-neighbour memorisation test. Image models use the same loop with a U-Net or transformer denoiser and
    millions of training images.
- **Where the frontier is.** Fast sampling (distillation into a few steps, consistency models — Song *et al.* 2023), understanding
    why diffusion models generalise instead of memorising their training sets (and when they do memorise; Carlini
    *et al.* 2023), diffusion in non-Euclidean spaces (rotations, torsion angles) and in discrete spaces (sequences),
    reliable likelihoods and uncertainty, and controllable generation.
- **Read next.** Ho, Jain & Abbeel 2020 (NeurIPS); Song *et al.* 2021 (ICLR); Karras *et al.* 2022 (NeurIPS).

---
## 5 · Flow matching: learning a velocity field

> 📍 **Where we are.** DDPM generates by a *random* walk of 1,000 small steps. **Flow matching** (Lipman *et al.* 2023;
> Liu, Gong & Liu 2023 — "rectified flow"; Albergo & Vanden-Eijnden 2023 — "stochastic interpolants") generates by a
> *deterministic* flow: every point is carried from noise to data by a learned velocity field, like a leaf on a
> stream. It is simpler to write down and usually needs far fewer steps.

**The construction.** Pair a noise point $x_0\sim\mathcal N(0,I)$ with a random data point $x_1$ and join them by a
straight line:

$$x_t=(1-t)\,x_0+t\,x_1,\qquad \frac{dx_t}{dt}=x_1-x_0,\qquad t\in[0,1].$$

*In words:* each pair moves at constant speed along a straight path from noise ($t=0$) to data ($t=1$). Train a
network $v_\theta(x,t)$ by plain regression on the velocity of those paths:

$$\mathcal L(\theta)=\mathbb E_{x_0,\,x_1,\,t}\Big[\big\lVert v_\theta(x_t,t)-(x_1-x_0)\big\rVert^2\Big].$$

*In words:* at a random point of a random path, predict which way and how fast the path is moving. Many paths pass
through any given $x$ (from different noise to different data points), so the best the network can do is predict their
**average** velocity, $v^*(x,t)=\mathbb E[x_1-x_0\mid x_t=x]$. The theorem behind the method is that this average field
transports the whole noise distribution at $t=0$ onto the data distribution at $t=1$. To generate, draw noise and
integrate the ordinary differential equation (ODE) $dx/dt=v_\theta(x,t)$ from 0 to 1.

🧮 **Math refresher — Euler's method.** To follow $dx/dt=v(x,t)$ numerically, chop $[0,1]$ into $K$ steps of size
$h=1/K$ and repeat $x\leftarrow x+h\,v(x,t)$, $t\leftarrow t+h$. It is exact if the velocity is constant along the
path; the more the paths curve, the more steps it needs. This is why *straight* paths matter.

> 🔗 **Builds on Sections 1 and 4.** Flow matching produces a normalising flow (Section 1) — an invertible map from
> noise to data, given by the ODE — without ever computing a Jacobian during training (a *continuous normalising flow*).
> And for Gaussian noise its velocity and the diffusion score are both simple (affine) functions of the same quantity,
> the expected data point $\mathbb E[x_1\mid x_t]$: the two families are close relatives (Albergo & Vanden-Eijnden 2023
> make this precise).

In [ ]:
# ── 5.1 Training a flow-matching model (the same network, a different target) ──
def train_flow_matching(X, labels=None, n_labels=0, n_steps=5000, batch=256, lr=1e-3, hidden=128, p_drop=0.2, seed=SEED):
    """Regress v_θ(x_t, t[, label]) on x_1 − x_0 along straight paths; labels are dropped with probability p_drop."""
    torch.manual_seed(seed)
    model = FourierMLP(dim=X.shape[1], hidden=hidden, n_labels=n_labels)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, n_steps)
    data, losses = torch.tensor(X, dtype=torch.float32), []
    lab_all = None if labels is None else torch.tensor(labels)
    for _ in range(n_steps):
        idx = torch.randint(0, len(data), (batch,))
        x1, x0, t = data[idx], torch.randn(batch, data.shape[1]), torch.rand(batch)
        x_t = (1 - t[:, None]) * x0 + t[:, None] * x1                 # a point on each straight path
        lab = None
        if n_labels:                                                  # label dropout (Section 6): sometimes "no label"
            lab = lab_all[idx].clone()
            lab[torch.rand(batch) < p_drop] = n_labels
        loss = ((model(x_t, t, lab) - (x1 - x0)) ** 2).mean()
        opt.zero_grad(); loss.backward(); opt.step(); sched.step()
        losses.append(loss.item())
    return model, np.array(losses)


@torch.no_grad()
def flow_sample(model, n, n_steps=20, label=None, guidance=1.0, seed=SEED, return_path=False):
    """Euler integration of dx/dt = v_θ(x, t) from noise (t = 0) to data (t = 1); optional classifier-free guidance."""
    g = torch.Generator().manual_seed(seed)
    x = torch.randn(n, model.dim, generator=g)
    path = [x.numpy().copy()]
    for k in range(n_steps):
        t = torch.full((n,), k / n_steps)
        if label is None:
            v = model(x, t)
        else:                                                         # guided velocity (Section 6)
            v_cond = model(x, t, torch.full((n,), label))
            v_null = model(x, t, torch.full((n,), model.n_labels))
            v = v_null + guidance * (v_cond - v_null)
        x = x + v / n_steps
        path.append(x.numpy().copy())
    return (x.numpy(), np.array(path)) if return_path else x.numpy()


t0 = time.time()
flow_fork, flow_losses = train_flow_matching(X_train)
print(f"flow matching trained in {time.time() - t0:.1f} s; loss {flow_losses[:100].mean():.3f} → "
      f"{flow_losses[-300:].mean():.3f} (it cannot reach 0: many paths cross every point)")

In [ ]:
# ── 5.2 🧪 Sample quality versus the number of network evaluations (NFE) ──
step_grid = [5, 10, 20, 50, 100, 1000]
quality = {"DDPM (ancestral)": [], "flow matching (Euler)": []}
t0 = time.time()
for K in step_grid:
    quality["DDPM (ancestral)"].append(energy_distance(ddpm_sample(ddpm_fork, 2000, n_steps=K, seed=SEED + K), X_test))
    quality["flow matching (Euler)"].append(energy_distance(flow_sample(flow_fork, 2000, n_steps=K, seed=SEED + K), X_test))
floor = report.loc["2,000 real training cells (noise floor)", "energy distance"]
print(f"evaluated in {time.time() - t0:.1f} s; noise floor = {floor:.4f}")
print(pd.DataFrame(quality, index=pd.Index(step_grid, name="steps")).round(4).T.to_string())
fig, ax = plt.subplots(figsize=(6.4, 3.4))
for (name, vals), col, mk in zip(quality.items(), [OKABE_ITO[0], OKABE_ITO[1]], "os"):
    ax.plot(step_grid, vals, marker=mk, color=col, label=name)
ax.axhline(floor, color="k", ls=":", lw=1, label="noise floor (real vs real)")
ax.set_xscale("log"); ax.set_yscale("log"); ax.set_xlabel("number of sampling steps (network evaluations)")
ax.set_ylabel("energy distance to held-out cells"); ax.legend(fontsize=7)
ax.set_title("Flow matching reaches good quality with far fewer steps")
plt.tight_layout(); plt.show()

In [ ]:
# ── 5.3 Paths from noise to data: straight flows versus random walks ──
_, path_fm = flow_sample(flow_fork, 25, n_steps=100, seed=SEED + 7, return_path=True)
_, path_dd = ddpm_sample(ddpm_fork, 25, n_steps=100, seed=SEED + 7, return_path=True)


def straightness(path):
    """|end − start| / path length for every trajectory (1 = perfectly straight)."""
    seg = np.linalg.norm(np.diff(path, axis=0), axis=-1).sum(0)
    return np.linalg.norm(path[-1] - path[0], axis=-1) / seg


fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharex=True, sharey=True)
for ax, (P, name) in zip(axes, [(path_fm, "flow matching (ODE, 100 Euler steps)"), (path_dd, "DDPM (ancestral, 100 steps)")]):
    ax.scatter(*X_test[:1500].T, s=1, color="0.85")
    for j in range(P.shape[1]):
        ax.plot(P[:, j, 0], P[:, j, 1], lw=0.7 if "flow" in name else 0.4,
                color=OKABE_ITO[0] if "flow" in name else OKABE_ITO[1], alpha=0.8 if "flow" in name else 0.5)
    ax.scatter(*P[0].T, s=8, color="k", zorder=3); ax.scatter(*P[-1].T, s=8, color=OKABE_ITO[2], zorder=3)
    ax.set_title(f"{name}\nmedian straightness = {np.median(straightness(P)):.2f}"); ax.set_aspect("equal")
    ax.set_xlim(-3.5, 3.5); ax.set_ylim(-3.5, 3.5)
plt.tight_layout(); plt.show()

🔍 **What just happened?** Read the table across. With 5 steps both samplers are poor, but flow matching is about three
times better (0.043 vs 0.125); at 10 and 20 steps it is four times better (0.0047 vs 0.021 at 20 steps, within a factor
of about four of the noise floor). DDPM needs about 50 or more steps to come close and gets nearest to the floor only
with all 1,000 steps (0.0018 against a floor of 0.0011) — where it is even slightly better than flow matching (0.0025). The rise of DDPM between 50 and 100
steps is run-to-run noise: repeating the sampling with other random seeds gives values between about 0.003 and 0.009 at
both settings. The paths explain the difference: the flow-matching trajectories cross the noise in nearly straight lines
and bend only near the end, where they settle onto a branch (median straightness about 0.7 — the average of many crossing
straight paths is not itself straight), so long Euler steps follow them well. DDPM's ancestral paths are random walks
that jitter all the way (straightness 0.05); their randomness is part of the algorithm, and long jumps lose accuracy.

**Sampling cost is the practical bottleneck of these models**: every step is a full network evaluation, and for a
protein-sized network one evaluation can take a sizeable fraction of a second on a GPU. Deterministic samplers (the
probability-flow ODE, DDIM), straighter couplings (minibatch optimal transport, Tong *et al.* 2024; "reflow", Liu, Gong &
Liu 2023) and distillation all aim to cut the number of steps.

⚠️ **Pitfall.** "Number of steps" is not the same as "time": a guided model (Section 6) needs two network evaluations
per step, and higher-order ODE solvers need several. Compare samplers by network evaluations (NFE).

✏️ **Your turn.** Replace the Euler step in `flow_sample` by Heun's method (evaluate $v$ at the start, take a trial
Euler step, evaluate $v$ at the end, and move with the average of the two). Does 10-step Heun (20 evaluations) beat
20-step Euler?

### 📄 Research connection — Flow matching and stochastic interpolants

- **What the field does.** In 2022 three groups independently proposed training continuous normalising flows by regressing the velocity
    of simple interpolation paths between noise and data, avoiding the expensive ODE simulation of earlier continuous
    flows: flow matching (Lipman *et al.* 2023), rectified flow (Liu, Gong & Liu 2023) and stochastic interpolants
    (Albergo & Vanden-Eijnden 2023). Tong *et al.* (2024) generalised the framework and showed that pairing noise and
    data by minibatch optimal transport yields straighter paths and faster sampling. Rectified-flow transformers are now
    used in large image generators (e.g. Esser *et al.* 2024).
- **What this notebook builds.** The straight-line (rectified-flow) version with independent noise–data pairs, an MLP velocity field, Euler
    integration, a comparison of quality against the number of network evaluations with DDPM, and a measure of path
    straightness. Section 6 adds labels and classifier-free guidance to the same model.
- **Where the frontier is.** Flow matching on curved spaces — rotations, torsion angles, protein frames (Riemannian flow matching;
    Chen & Lipman 2024) — which underlies several protein-backbone generators released since 2023; few-step and
    one-step generation; couplings between *two* data distributions (e.g. unperturbed and perturbed cells, the
    optimal-transport problems of Module 32); and theory of when the learned flow generalises.
- **Read next.** Lipman *et al.* 2023 (ICLR); Liu, Gong & Liu 2023 (ICLR); Tong *et al.* 2024 (TMLR).

---
## 6 · Conditional generation and guidance: cells on request

> 📍 **Where we are.** An unconditional model generates *some* cell. Biology usually wants a *particular* one: a cell of
> a given type, a protein binding a given target, a response to a given drug. Here we generate single cells of a
> requested type and learn the most widely used control knob, **classifier-free guidance**.

**Conditioning.** Give the network the label $y$ as an extra input and train on (cell, type) pairs: $v_\theta(x,t,y)$ then
transports noise to $p(x\mid y)$. Classifier-free guidance (Ho & Salimans 2022) adds one trick: during training, replace the
label by a special "no label" token for a random 20 % of examples (**label dropout**), so that the *same* network also
learns the unconditional field $v_\theta(x,t,\varnothing)$. At sampling time, combine the two:

$$v_\gamma(x,t,y)=v_\theta(x,t,\varnothing)+\gamma\,\big[v_\theta(x,t,y)-v_\theta(x,t,\varnothing)\big].$$

*In words:* start from the direction that leads to *any* cell, and add $\gamma$ times the extra push that leads to a cell
*of type $y$*. $\gamma=0$ is unconditional, $\gamma=1$ is the plain conditional model, and $\gamma>1$ exaggerates what
distinguishes type $y$ from the rest. (Ho & Salimans write the weight as $w=\gamma-1$.) In score language,
$\gamma\nabla\log p(x\mid y)+(1-\gamma)\nabla\log p(x)=\nabla\log p(x)+\gamma\,\nabla\log p(y\mid x)$: guidance raises an
implicit classifier $p(y\mid x)$ to the power $\gamma$, favouring cells that are *unmistakably* of type $y$ — and, as we
will measure, giving up the less typical ones. (The older *classifier guidance* of Dhariwal & Nichol 2021 used a separately
trained classifier's gradient instead.)

🔬 **Assay box — what the simulated cells stand for.** *What is measured:* in droplet single-cell RNA-seq each cell is
captured in a droplet with a barcoded bead; its mRNA molecules are reverse-transcribed with a cell barcode and a unique
molecular identifier (UMI), so that sequencing yields a cells × genes table of molecule counts. *Typical sizes:*
$10^3$–$10^6$ cells, ~20,000 genes, a few hundred to several thousand UMIs per cell, most entries zero. *Typical
artefacts:* variable capture depth, ambient RNA, doublets, damaged cells and batch effects. *Perturbation screens*
(Perturb-seq) add a CRISPR guide barcode per cell, so each profile is labelled with the gene that was knocked down — the
kind of data a "virtual cell" generator would be trained on. The full assay boxes are in
[Module 25](25_variational_inference_vae.ipynb) and [Module 30](30_single_cell_rnaseq.ipynb).

**The data.** 3,000 SIMULATED PBMC-like cells (`simulate_scrna`, one batch, six cell types; Module 25 introduced the
simulator), log-normalised and reduced to 20 principal components
([Module 14](14_pca_dimensionality_reduction.ipynb)). We simulate shallow sequencing (about 300 UMIs per cell) so that
similar types — CD4 and CD8 T cells — overlap a little, as they do in real data. The generator works in PCA space; the
PCA loadings decode its output back into genes.

In [ ]:
# ── 6.1 SIMULATED PBMC-like cells in a 20-D PCA space, and a classifier trained on real cells ──
rng = np.random.default_rng(SEED + 6)
sim = simulate_scrna(rng, n_cells=3000, n_genes=800, n_batches=1, depth=300)
cell_type = sim["obs"]["true_type"].to_numpy(str)
TYPES = np.unique(cell_type)
y_cells = np.searchsorted(TYPES, cell_type)
X_log = log_normalize(sim["counts"])
gene_mean = X_log.mean(0)
pc_scores, pc_loadings, pc_var = pca(X_log, 20)
pc_center, pc_scale = pc_scores.mean(0), pc_scores.std()          # one global scale keeps the geometry
Z_cells = (pc_scores - pc_center) / pc_scale
order = rng.permutation(len(Z_cells))
tr, te = order[:2400], order[2400:]
clf = LogisticRegression(max_iter=3000).fit(Z_cells[tr], y_cells[tr])
print(f"SIMULATED: {len(Z_cells)} cells × 800 genes, median {np.median(sim['counts'].sum(1)):.0f} UMIs per cell; "
      f"20 PCs keep {100 * pc_var.sum():.1f} % of the variance")
print(f"classifier (logistic regression on 20 PCs), accuracy on held-out REAL cells: {clf.score(Z_cells[te], y_cells[te]):.3f}")
conf = pd.crosstab(pd.Series(TYPES[y_cells[te]], name="true"), pd.Series(TYPES[clf.predict(Z_cells[te])], name="predicted"))
print(conf.loc[["CD4 T", "CD8 T"], ["CD4 T", "CD8 T"]].to_string())
fig, ax = plt.subplots(figsize=(5, 3.8))
plot_embedding(Z_cells[:, :2], cell_type, ax=ax, title="SIMULATED cells: PC 1 vs PC 2 (true types)")
plt.tight_layout(); plt.show()

In [ ]:
# ── 6.2 A class-conditional flow-matching model with label dropout ──
t0 = time.time()
flow_cells, _ = train_flow_matching(Z_cells[tr], labels=y_cells[tr], n_labels=len(TYPES), n_steps=3000, p_drop=0.2)
print(f"conditional flow matching on {len(tr)} cells × 20 dimensions trained in {time.time() - t0:.1f} s")


def precision_recall(G, R, k=5):
    """k-NN precision (generated cells inside the real data's k-NN balls) and recall (real cells inside the generated
    cells' balls) — Kynkäänniemi et al. 2019."""
    r_real, r_gen = np.sort(cdist(R, R), 1)[:, k], np.sort(cdist(G, G), 1)[:, k]
    D = cdist(G, R)
    return (D <= r_real[None]).any(1).mean(), (D.T <= r_gen[None]).any(1).mean()

In [ ]:
# ── 6.3 🧪 Guidance sweep: does the generated cell have the requested type, and how diverse are the cells? ──
GAMMAS = [0, 1, 1.5, 2, 3, 5]
generated, sweep = {}, []
for gamma in GAMMAS:
    acc, prec, rec, edist, spread = [], [], [], [], []
    for k in range(len(TYPES)):
        G = flow_sample(flow_cells, 200, n_steps=40, label=k, guidance=gamma, seed=SEED + k)
        generated[(gamma, k)] = G
        R = Z_cells[te][y_cells[te] == k]                              # held-out real cells of that type
        acc.append(np.mean(clf.predict(G) == k))
        p, r = precision_recall(G, R); prec.append(p); rec.append(r)
        edist.append(energy_distance(G, R))
        spread.append(G.var(0).sum() / Z_cells[tr][y_cells[tr] == k].var(0).sum())
    sweep.append({"guidance γ": gamma, "requested type %": 100 * np.mean(acc), "precision": np.mean(prec),
                  "recall": np.mean(rec), "spread / real": np.mean(spread), "energy distance": np.mean(edist)})
sweep = pd.DataFrame(sweep).set_index("guidance γ")
print("averages over the six types; 200 generated cells per type; real held-out cells as reference")
print(sweep.round(3).to_string())

In [ ]:
# ── 6.4 Picture: the fidelity–diversity trade-off ──
fig, axes = plt.subplots(1, 3, figsize=(12, 3.6), gridspec_kw={"width_ratios": [1, 1, 1.1]})
for ax, gamma in zip(axes[:2], (1, 5)):
    ax.scatter(*Z_cells[te][:, :2].T, s=3, color="0.85", label="real (held-out)")
    for k in range(len(TYPES)):
        ax.scatter(*generated[(gamma, k)][:, :2].T, s=3, color=plt.get_cmap("tab10")(k), label=TYPES[k])
    ax.set_title(f"generated cells, guidance γ = {gamma}"); ax.set_xticks([]); ax.set_yticks([])
    ax.set_xlabel("PC 1"); ax.set_ylabel("PC 2")
axes[0].legend(markerscale=3, fontsize=6, loc="best")
ax = axes[2]
ax.plot(GAMMAS, sweep["requested type %"] / 100, "o-", color=OKABE_ITO[0], label="requested type (classifier)")
ax.plot(GAMMAS, sweep["precision"], "s-", color=OKABE_ITO[2], label="precision (realistic)")
ax.plot(GAMMAS, sweep["recall"], "^-", color=OKABE_ITO[1], label="recall (covers real variety)")
ax.set_xlabel("guidance γ"); ax.set_ylabel("fraction"); ax.set_ylim(0, 1.05); ax.legend(fontsize=7)
ax.set_title("guidance: the type is always right, but variety shrinks")
plt.tight_layout(); plt.show()

🔍 **What just happened?** Read the table from the top.

* **γ = 0** ignores the label: about one sixth of the cells (17 %) happen to have the requested type, and the spread is
  that of the whole data set (1.6 times one type's spread).
* **γ = 1**, the plain conditional model, generates the requested type as reliably as the classifier recognises real
  cells (98.9 % vs 98.3 %; on real cells the classifier's mistakes are CD4/CD8 confusions), with precision 0.92, recall
  0.76 and the smallest
  energy distance (0.083) — the most faithful copy of the real distribution.
* **γ > 1** makes the type *more unmistakable than in real data* (100 % classified correctly) and at first even slightly
  more realistic (precision 0.94 at γ = 1.5–2), but recall falls steadily — to 0.32 at γ = 5 — the spread shrinks to
  about 0.78 of the real one and the energy distance grows eightfold. The guided cells are caricatures of their type,
  concentrated in its most typical region and pushed away from neighbouring types (visible as thin streaks in the γ = 5
  panel). At γ = 5 precision drops too (0.82), because the exaggeration overshoots the real data.

This is the **fidelity–diversity trade-off** of guidance, and it matters for biology: a generator used to propose
cells, proteins or molecules with γ > 1 will under-represent rare and intermediate states. In image generation large
γ is popular because people prefer typical, crisp pictures; in science, faithful coverage often matters more.

The last check leaves PCA space: do the generated cells have the right *genes*? We decode them with the PCA loadings and
compare each type's marker genes with real cells.

In [ ]:
# ── 6.5 🧪 Decode generated cells into genes: marker genes and type-specific profiles ──
markers = sim["var"]["marker_of"].to_numpy(str)
genes = sim["var"].index.to_numpy(str)


def to_genes(Z):
    """Map scaled PCA coordinates back to log-normalised expression of all 800 genes."""
    return (Z * pc_scale + pc_center) @ pc_loadings + gene_mean


def marker_lfc(X, is_type, named):
    """Mean log-expression of the named marker genes in cells of the type minus in all other cells."""
    return X[is_type][:, named].mean() - X[~is_type][:, named].mean()


X_real_te, X_recon_te = X_log[te], to_genes(Z_cells[te])          # real cells, and the same cells seen through 20 PCs
rows6 = []
for k, name in enumerate(TYPES):
    named = np.isin(genes, [g for g, m in zip(genes, markers) if m == name][:4])      # the type's 4 named markers
    is_k = y_cells[te] == k
    row = {"type": name, "real": marker_lfc(X_real_te, is_k, named), "real via 20 PCs": marker_lfc(X_recon_te, is_k, named)}
    dev_real = X_recon_te[is_k].mean(0) - X_recon_te.mean(0)                          # type-specific profile (real)
    for gamma in (1, 5):
        G_all = np.vstack([to_genes(generated[(gamma, j)]) for j in range(len(TYPES))])
        is_gen = np.repeat(np.arange(len(TYPES)), 200) == k
        row[f"generated γ={gamma}"] = marker_lfc(G_all, is_gen, named)
        dev_gen = G_all[is_gen].mean(0) - G_all.mean(0)
        row[f"r(profile) γ={gamma}"] = np.corrcoef(dev_gen, dev_real)[0, 1]
    rows6.append(row)
print("marker log-fold change: mean log-expression of the type's 4 named markers in that type minus in the other types;")
print("r(profile): correlation over 800 genes of the type's deviation from the average cell, generated vs real (via 20 PCs)")
print(pd.DataFrame(rows6).set_index("type").round(3).to_string())

🔍 **What just happened?** The fair target is the second column: real cells *seen through the same 20 PCs* the generator
works in (20 components cannot carry every gene's detail, so even real cells lose some marker contrast, e.g. CD8 T 1.17 →
0.91). Against that target the conditional cells at γ = 1 are excellent: every type's marker log-fold change is within
about 0.07 of it, and the type-specific profile over all 800 genes correlates at r = 0.97–0.99. At γ = 5 the profiles
correlate less well (r = 0.91–0.97) and the markers are distorted — **exaggerated** for the B, T and dendritic cells that
have close neighbours (CD4 T 0.43 instead of 0.20, CD8 T 1.66 instead of 0.91), while for monocytes and NK cells, which
are already far from everything else, guidance pushes along other directions and their markers even fall slightly. That
is the gene-level face of the caricature effect: guided cells are not "better" cells, they are cells pushed away from the
other classes.

⚠️ **Pitfall — generating in a reduced space.** We generated PCA coordinates, not counts. Real single-cell generators
either model counts directly (e.g. a negative-binomial decoder, as in the VAE of
[Module 25](25_variational_inference_vae.ipynb)) or generate in a learned latent space and decode; evaluating in the
generator's own space alone can hide problems that appear in gene space.

✏️ **Your turn.** Retrain the conditional model with `p_drop=0.0`. Why does guidance stop making sense? Then try
`p_drop=0.5`: what happens to the accuracy at γ = 1?

### 📄 Research connection — Generative models of cells and perturbations

- **What the field does.** Deep generative models of single cells began with VAEs — scVI for counts (Lopez *et al.* 2018), scGen and
    CPA for predicting perturbation responses by arithmetic in a latent space (Lotfollahi *et al.* 2019, 2023) — and
    optimal-transport methods that map unperturbed to perturbed cell populations (CellOT; Bunne *et al.* 2023). Since
    about 2023, diffusion and flow-matching models have been applied to generate expression profiles, augment rare
    cell types and predict perturbation responses, usually in a latent or PCA space as here.
- **What this notebook builds.** A class-conditional flow-matching generator in a 20-D PCA space of simulated PBMC-like cells, with label
    dropout and classifier-free guidance, evaluated against held-out real cells by a classifier, k-NN precision and
    recall, energy distance and marker genes.
- **Where the frontier is.** The idea of a "virtual cell" that predicts unseen states and responses is a stated goal of the field
    (Bunne *et al.* 2024), but evaluation is hard: benchmarks have found that, for predicting the effects of genetic
    perturbations, current deep models often do not beat simple linear baselines (Ahlmann-Eltze, Huber & Anders
    2025). Open problems include generating raw counts faithfully, conditioning on combinations never seen
    in training, and measuring diversity and novelty rather than only typicality.
- **Read next.** Lotfollahi *et al.* 2019 (Nat Methods); Bunne *et al.* 2023 (Nat Methods); Bunne *et al.* 2024 (Cell).

---
## 7 · Diffusion beyond points: structures, molecules and sequences

> 📍 **Where we are.** Everything so far lived in a flat space where each coordinate is a number. Proteins and molecules
> are point clouds in 3-D with symmetries, and sequences are strings of discrete letters. This section explains,
> conceptually and with two small demonstrations, how diffusion is adapted to them.

**Noising a protein backbone.** Treat the $L$ Cα atoms of a protein as one vector in $\mathbb R^{3L}$, centre it (the
absolute position is meaningless), and apply exactly the forward process of Section 3 to the coordinates. The
denoiser receives a cloud of noisy atoms and must predict the noise — i.e. guess, from a blurred cloud, where a real
protein's atoms would be.

In [ ]:
# ── 7.1 Forward diffusion of a protein backbone (the SIMULATED four-helix bundle) ──
bundle = simulate_helix_bundle()
CA = bundle["coords"] - bundle["coords"].mean(0)              # 68 Cα atoms, centred (Å)
SCALE_A = 10.0                                                 # work in units of 10 Å so coordinates are O(1)
native_contacts = bundle["contacts"]
rng = np.random.default_rng(SEED + 7)
eps_ca = rng.standard_normal(CA.shape)
fig, axes = plt.subplots(1, 6, figsize=(12, 2.5), sharex=True, sharey=True)
print(" step t | noise in Å, σ_eff | RMSD to native (Å): measured  expected σ√3 | native contacts kept | Cα–Cα steps off by > 0.5 Å")
for ax, t in zip(axes, (0, 25, 50, 100, 200, 400)):
    ab = 1.0 if t == 0 else ABAR[t - 1]
    x_t = np.sqrt(ab) * CA / SCALE_A + np.sqrt(1 - ab) * eps_ca          # q(x_t | x_0) on the coordinates
    view = x_t / np.sqrt(ab) * SCALE_A                                    # undo the shrinking to compare in Å
    sigma_eff = SCALE_A * np.sqrt((1 - ab) / ab)
    D = np.linalg.norm(view[:, None] - view[None], axis=-1)
    kept = np.mean(D[native_contacts] < 8.0)
    print(f"  {t:5d} | {sigma_eff:8.2f}          | {np.sqrt(((view - CA) ** 2).sum(1).mean()):8.2f}   "
          f"{sigma_eff * np.sqrt(3):8.2f}       |   {100 * kept:5.1f} %          | "
          f"{100 * np.mean(np.abs(np.linalg.norm(np.diff(view, axis=0), axis=1) - 3.8) > 0.5):5.1f} %")
    ax.plot(x_t[:, 0], x_t[:, 2], color="0.6", lw=0.6)
    ax.scatter(x_t[:, 0], x_t[:, 2], c=np.arange(len(CA)), cmap="viridis", s=8)
    ax.set_title(f"t = {t}", fontsize=8); ax.set_aspect("equal"); ax.set_xticks([]); ax.set_yticks([])
fig.suptitle("Forward diffusion of a SIMULATED Cα backbone (side view; colour = residue number)", fontsize=10)
plt.tight_layout(); plt.show()
print(f"(the same noise draw is reused at every t; its 204 numbers have mean square {np.mean(eps_ca ** 2):.2f} instead of 1)")

🔍 **What just happened?** The measured RMSD follows the prediction $\sigma_{\text{eff}}\sqrt3$ (three coordinates per
atom, each with noise $\sigma_{\text{eff}}$) at every step, up to a constant factor of 0.88: we reuse one random draw of
204 numbers whose mean square happens to be 0.78 rather than 1, and $\sqrt{0.78}=0.88$ (averaged over many draws the
prediction is exact). Already at $t=25$ (just under 1 Å of noise) the 3.8 Å Cα–Cα spacing — a hard physical constraint —
is broken: 73 % of the steps between neighbouring Cα atoms are off by more than 0.5 Å, although 90 % of the native
contacts are still in place. By $t=100$ (3.4 Å of noise) fewer than half of the contacts survive (48 %), by $t=200$ the
helices are gone (15 %), and at $t=400$ (20 Å of noise) the chain is a random cloud. A structure generator must
learn to recreate all of this — bond geometry, secondary structure, packing — from noise.

**Why 3-D generation needs equivariance.** A protein rotated in space is the same protein. If the noisy input is rotated,
the predicted noise (or denoised structure) should rotate with it — the denoiser should be **rotation-equivariant**
([Module 28](28_graph_neural_networks.ipynb), Sections 7–8, built and verified an E(n)-equivariant layer). There are
two strategies:

* **Build it in.** Equivariant networks (EGNN-like layers, invariant point attention) act on residue frames or
  coordinates so that equivariance holds exactly. RFdiffusion (Watson *et al.* 2023) diffuses residue *frames* — a
  position (Gaussian diffusion as above) and an orientation (diffusion on the space of rotations) — and uses the
  fine-tuned RoseTTAFold structure-prediction network as the denoiser.
* **Learn it from data.** AlphaFold 3's diffusion module (Abramson *et al.* 2024) generates the coordinates of *all
  atoms* of a complex with a non-equivariant transformer, conditioned on the representations of its trunk; the authors
  report that random rotations and translations of the training structures (data augmentation) were enough.

**Other variations.** Chroma (Ingraham *et al.* 2023) uses noise with the correlations of a random polymer chain rather
than independent noise per atom, and steers generation with "conditioners" (symmetry, shape, even text). DiffDock
(Corso *et al.* 2023) docks a small molecule into a protein by diffusing only the ligand's degrees of freedom: its
position, its orientation and its rotatable torsion angles.

**Discrete diffusion for sequences.** Sequences have no Gaussian noise; the most common replacement is the *absorbing*
or **masking** process (Austin *et al.* 2021): at noise level $t\in[0,1]$, every letter is independently hidden with
probability $t$.

In [ ]:
# ── 7.2 Masking diffusion: the forward process for a protein sequence ──
rng = np.random.default_rng(SEED + 72)
seq = "".join(rng.choice(list(AMINO_ACIDS), 40))          # a SIMULATED 40-residue sequence
u = rng.random(len(seq))                                  # one uniform number per residue: hidden once t > u
print("masking diffusion of a SIMULATED sequence (· = hidden):")
for t in (0.0, 0.25, 0.5, 0.75, 1.0):
    noisy = "".join("·" if u_i < t else a for a, u_i in zip(seq, u))
    print(f"t = {t:4.2f}  {noisy}   ({np.sum(u < t):2d}/40 hidden)")

🔍 **What just happened?** As $t$ grows, more letters are hidden (a hidden letter stays hidden), until at $t=1$ nothing
is left — the discrete counterpart of pure noise. The **denoiser** of this process predicts the hidden letters from the
visible ones: it is exactly the *masked language model* of [Module 26](26_attention_transformers.ipynb), trained at all
masking rates instead of a fixed 15 %. Generation starts from a fully masked sequence and unmasks a few positions at a
time. Protein language models trained with variable masking rates, such as ESM3 (Hayes *et al.* 2025), can therefore
generate sequences in this iterative way.

> 🔗 **Builds on [Module 26](26_attention_transformers.ipynb), Section 4.** Masked-language-model pre-training is one
> step of a discrete diffusion model; a masked LM used generatively is a discrete diffusion sampler.

✏️ **Your turn.** In 7.1, replace the independent noise `eps_ca` by noise that is correlated along the chain
(e.g. a cumulative sum of small random steps, re-centred and rescaled to unit variance per coordinate). Which property
of the backbone survives longer — the Cα–Cα spacing or the contacts? (This is the intuition behind Chroma's polymer noise.)

### 📄 Research connection — Generative models in structural biology

- **What the field does.** Diffusion models became the dominant approach to generating protein structures within a few years.
    RFdiffusion (Watson *et al.* 2023) generated backbones for binders, symmetric assemblies and enzyme-like motif
    scaffolds, with many designs validated experimentally (for example, a cryo-EM structure of a designed binder bound
    to its target closely matched the design). Chroma (Ingraham *et al.* 2023) introduced polymer-structured noise and
    programmable conditioning. DiffDock (Corso *et al.* 2023) cast molecular docking as diffusion over ligand poses,
    and equivariant diffusion was developed for small molecules in 3-D (Hoogeboom *et al.* 2022). AlphaFold 3
    (Abramson *et al.* 2024) uses a diffusion module to predict all-atom structures of complexes of proteins, nucleic
    acids, ligands and ions.
- **What this notebook builds.** The forward process on a Cα backbone (with the RMSD and contact-loss checks), the masking forward process
    for sequences, and — in the rest of the notebook — the core algorithms these models share: noise schedules, learned
    denoisers, ancestral and ODE sampling, conditioning and guidance.
- **Where the frontier is.** All-atom design of proteins together with small molecules and nucleic acids (e.g. RoseTTAFold All-Atom;
    Krishna *et al.* 2024), joint generation of sequence and structure, generating conformational ensembles rather than
    single structures, flow matching on residue frames, honest evaluation (designability, novelty, experimental success
    rates; [Module 39](39_generative_design.ipynb)), the tendency of structure generators to produce plausible-looking
    structure in disordered regions, and biosecurity considerations of open design tools.
- **Read next.** Watson *et al.* 2023 (Nature); Ingraham *et al.* 2023 (Nature); Abramson *et al.* 2024 (Nature).

---
## 8 · Evaluation and pitfalls

> 📍 **Where we are.** Generative models are easy to make and hard to evaluate. This section provokes the most
> important failure — memorisation — and collects the questions to ask of any generated sample.

**A thought experiment.** Suppose a flow-matching model fitted its training objective *perfectly* on a training set of
$N$ points $\{y_k\}$. The best possible velocity field for this finite set has a closed form: given $x_t$, the
probability that the path came from training point $y_k$ is a softmax,

$$w_k(x,t)\propto\exp\!\Big(-\frac{\lVert x-t\,y_k\rVert^2}{2(1-t)^2}\Big),\qquad
v^*(x,t)=\frac{\sum_k w_k(x,t)\,y_k-x}{1-t}.$$

*In words:* the perfect velocity points towards a weighted average of the training points, with weights that become
one-hot as $t\to1$. Following it must therefore end *exactly on a training point*: **the perfect fit to the training
data is a perfect memoriser.** A neural network generalises only because it does *not* fit this field perfectly — its
smoothness and limited training act as regularisation. Let us compare the two on a deliberately small training set
of 100 cells.

In [ ]:
# ── 8.1 🧪 The perfect memoriser versus a neural network, on 100 training cells ──
small = X_train[:100]


def empirical_velocity(x, t, Y):
    """The exact minimiser of the flow-matching loss for the finite training set Y (rows), at points x and time t < 1."""
    logw = -((x[:, None, :] - t * Y[None]) ** 2).sum(-1) / (2 * (1 - t) ** 2)
    w = np.exp(logw - lse(logw, axis=1, keepdims=True))
    return (w @ Y - x) / (1 - t)


rng = np.random.default_rng(SEED + 8)
x_exact = rng.standard_normal((1000, 2))
for k in range(100):                                       # Euler integration with the closed-form field
    x_exact = x_exact + empirical_velocity(x_exact, k / 100, small) / 100
t0 = time.time()
flow_small, _ = train_flow_matching(small, n_steps=3000)
x_neural = flow_sample(flow_small, 1000, n_steps=100, seed=SEED + 8)
print(f"neural flow matching on 100 cells trained in {time.time() - t0:.1f} s")
nn_exact, nn_neural = cdist(x_exact, small).min(1), cdist(x_neural, small).min(1)
nn_new = cdist(X_test[:1000], small).min(1)
for name, d in [("perfect (closed-form) model", nn_exact), ("neural network", nn_neural), ("new real cells", nn_new)]:
    print(f"{name:28s}: median distance to nearest training cell {np.median(d):.1e}; within 0.01: {100 * np.mean(d < 0.01):5.1f} %")
print(f"distinct training cells reproduced by the perfect model: {len(np.unique(cdist(x_exact, small).argmin(1)))} of 100")

In [ ]:
# ── 8.2 Picture: copying versus generalising ──
fig, axes = plt.subplots(1, 3, figsize=(12, 3.6), gridspec_kw={"width_ratios": [1, 1, 1.2]})
for ax, (S, name, col) in zip(axes[:2], [(x_exact, "perfect fit: 1,000 samples", OKABE_ITO[1]),
                                         (x_neural, "neural network: 1,000 samples", OKABE_ITO[0])]):
    ax.scatter(*S.T, s=4, color=col, alpha=0.5, label="generated")
    ax.scatter(*small.T, s=18, facecolor="none", edgecolor="k", lw=0.8, label="100 training cells")
    ax.set_title(name); ax.set_aspect("equal"); ax.legend(fontsize=7, loc="upper center")
bins = np.linspace(0, 0.3, 50)
axes[2].hist(nn_new, bins=bins, alpha=0.5, density=True, color="0.5", label="new real cells")
axes[2].hist(nn_neural, bins=bins, alpha=0.5, density=True, color=OKABE_ITO[0], label="neural network")
axes[2].axvline(np.median(nn_exact), color=OKABE_ITO[1], lw=2, label="perfect fit (all ≈ 0)")
axes[2].set_xlabel("distance to nearest training cell"); axes[2].legend(fontsize=7)
axes[2].set_title("nearest-neighbour test")
plt.tight_layout(); plt.show()

🔍 **What just happened?** The closed-form "perfect" model returned copies: 98 % of its 1,000 samples lie within 0.01 of
a training cell (median distance $3\times10^{-16}$ — numerically zero), and between them they reproduce all 100 training
cells. By the energy
distance alone it would look excellent — it reproduces the training distribution exactly — yet it has learned nothing
new. The neural network trained on the very same 100 cells generates cells that are as far from the training cells as
new real cells are: it fills in the fork between and around the training examples. **Generalisation in diffusion and
flow models comes from the network's inductive bias, not from the objective**, and with enough capacity and training
time on small data sets, real models do memorise: Carlini *et al.* (2023) extracted near-copies of training images
from large image diffusion models.

**What to ask of any generated sample** (for cells, proteins or molecules):

| Question | Our measurement | Real-world counterpart |
|---|---|---|
| Is it realistic? (fidelity) | off-fork %, precision, classifier | FID for images; for proteins: is the designed structure predicted to fold (AlphaFold/ESMFold confidence)? |
| Does the set cover the real variety? (diversity) | branch shares, recall, spread | number of distinct clusters; pairwise TM-scores within a design batch |
| Is it new? (novelty) | nearest-neighbour distances | similarity to the closest PDB entry or training sequence |
| Does it satisfy the request? (conditioning) | requested-type accuracy, marker genes | binding to the target, motif present, measured activity |
| What did it cost? | network evaluations | GPU-seconds per design |

For protein design the standard in-silico check is **designability** (self-consistency): design a backbone, design a
sequence for it, refold the sequence with a structure predictor and require the refolded structure to match the design
(for example, an RMSD below 2 Å) — [Module 39](39_generative_design.ipynb) discusses this pipeline and builds its
inverse-folding step in miniature, and [Module 38](38_protein_structure.ipynb) builds the structure-comparison tools it needs.

⚠️ **Pitfalls collected.**
1. **Memorisation** looks like success on most quality metrics; always run a nearest-neighbour test against the training set.
2. **Mode collapse and over-guidance** silently drop rare states (Section 6: recall fell steadily as γ grew).
3. **The wrong yardstick**: quality in the generator's working space (PCA, latent) is not quality in gene or atom space.
4. **In-silico success is a hypothesis**, not a result: generated cells and proteins become evidence only after experiments.

✏️ **Your turn.** Train the neural model of 8.1 on only 20 training cells and for 10,000 steps. Does it start to
memorise? Measure it with the nearest-neighbour test.

---
## 9 · 🧭 From toy to real data

> 📍 **Where we are.** Our models had fewer than 50,000 parameters and trained in seconds on a CPU. This section is reading: what
> changes for real proteins and cells.

**Scale and compute.** Image and protein diffusion models have tens of millions to billions of parameters and train for
days to weeks on many GPUs; sampling one protein backbone with a model such as RFdiffusion takes seconds to minutes on a
single GPU. Single-cell generators are smaller but are trained on millions of cells. Everything in this notebook
transfers; what changes is the denoiser (transformers, equivariant networks, U-Nets), the space (frames, all atoms,
latent spaces, counts) and the evaluation.

**Public data and resources.**
[PDB](https://www.rcsb.org) (experimental structures — the training data of protein structure generators),
[AlphaFold DB](https://alphafold.ebi.ac.uk) (predicted structures, used for training and as a reference for novelty),
[CATH](https://www.cathdb.info) (domain classification, used to build non-redundant training sets),
[CELLxGENE](https://cellxgene.cziscience.com/) and the [Human Cell Atlas](https://www.humancellatlas.org/) (single-cell
data at scale), and the Replogle *et al.* (2022) genome-wide Perturb-seq screens (perturbation responses; Module 41).

**Production tools.** Open-source protein generators include RFdiffusion (with ProteinMPNN for sequence design and
AlphaFold 2 or ESMFold for checking designs) and Chroma; AlphaFold 3 code and weights are available for academic use and
several open models of the same type appeared in 2024–2025 (e.g. Boltz-1, Chai-1); DiffDock for docking. For writing
your own models: PyTorch, plus the Hugging Face `diffusers` library for standard schedulers and samplers.

**The first three things that go wrong.**

1. **Training-set leakage and redundancy.** Proteins come in families; a "novel" design that is a near-copy of a
   training protein, or a test set that shares families with the training set, inflates performance. Cluster by
   sequence or structure similarity before splitting ([Module 42](42_trustworthy_ml.ipynb)).
2. **Evaluation with the generator's own tools.** Designs filtered by the same structure predictor that helped generate
   them can look better than they are; experimental success rates, not in-silico scores, are the ground truth.
3. **Conditioning outside the data.** Asking for a cell type, a perturbation or a protein shape that the training data
   barely contain produces confident-looking samples with no support — the generative version of extrapolation.

✏️ **Your turn.** Pick one published generative model for proteins or cells. From its paper, find (a) the training data
and how it was split, (b) the sampling procedure and number of steps, (c) how samples were evaluated and whether any
were tested experimentally. Which row of the table in Section 8 does it address least?

---
## 📌 Key takeaways

- A **generative model** learns $p(x)$ to *sample* new examples; mixtures, VAEs, GANs, flows, diffusion and flow
  matching trade off sampling speed, exact densities and flexibility. The **change-of-variables formula** links noise to
  data through a transformation.
- The **score** $\nabla\log p$ needs no normalising constant; **Langevin dynamics** samples with it, but plain Langevin
  got the proportions of two separated states wrong (≈ 50 / 50 instead of 80 / 20) — **annealing over noise levels**
  largely fixed it (≈ 77 / 23). Diffusion models learn exactly these noise-level-dependent scores.
- The **forward diffusion** has the closed form $q(x_t\mid x_0)=\mathcal N(\sqrt{\bar\alpha_t}x_0,(1-\bar\alpha_t)I)$;
  training a **DDPM** is regression of the added noise, and the trained denoiser estimates the score
  ($s\approx-\varepsilon_\theta/\sqrt{1-\bar\alpha_t}$; checked against an exact score).
- **Flow matching** regresses the velocity $x_1-x_0$ of straight noise-to-data paths and samples with an ODE; here it
  was 3–4× better than DDPM at every budget of 5–20 steps, while DDPM needed 50–1,000 steps for comparable quality.
- **Classifier-free guidance** (label dropout + $v_\varnothing+\gamma(v_y-v_\varnothing)$) trades diversity for
  fidelity: generated cells became unmistakably of the requested type while recall fell from 0.76 to 0.32.
- **Evaluate generators against held-out data and the training set**: energy distance, precision/recall, branch shares,
  marker genes, and a nearest-neighbour test — because **a perfect fit to the training data is a memoriser**.
- The same algorithms generate **protein backbones, complexes and docked poses** (RFdiffusion, Chroma, AlphaFold 3,
  DiffDock) once the denoiser respects 3-D symmetry; **masking diffusion** connects them to protein language models.

---
## ✅ Checkpoint

1. Starting from $x_1=\sqrt{\alpha_1}x_0+\sqrt{\beta_1}\varepsilon_1$ and $x_2=\sqrt{\alpha_2}x_1+\sqrt{\beta_2}\varepsilon_2$, show that $x_2\mid x_0\sim\mathcal N(\sqrt{\alpha_1\alpha_2}\,x_0,(1-\alpha_1\alpha_2)I)$.
   <details><summary>Answer</summary>Substitute: $x_2=\sqrt{\alpha_1\alpha_2}x_0+\sqrt{\alpha_2\beta_1}\varepsilon_1+\sqrt{\beta_2}\varepsilon_2$. The two noise terms are independent Gaussians, so their sum is Gaussian with variance $\alpha_2\beta_1+\beta_2=\alpha_2(1-\alpha_1)+1-\alpha_2=1-\alpha_1\alpha_2$.</details>
2. Why do DDPMs usually predict the noise $\varepsilon$ rather than the clean data $x_0$, and how is the predicted noise related to the score?
   <details><summary>Answer</summary>The noise has unit variance at every step, so the regression target is equally well scaled for all $t$ (predicting $x_0$ is equivalent but weights noise levels differently and trained worse in Ho et al. 2020). The best prediction is the average noise given $x_t$, and $\nabla\log q_t(x_t)=-\mathbb E[\varepsilon\mid x_t]/\sqrt{1-\bar\alpha_t}$ — so $-\varepsilon_\theta/\sqrt{1-\bar\alpha_t}$ estimates the score (verified in 4.3).</details>
3. By hand: a mixture has two components with means $-3$ and $+3$ (variance 0.25, 1-D). At $x=-2.5$ the responsibilities are essentially $(1,0)$. What is the score, and in which direction does a Langevin step move?
   <details><summary>Answer</summary>$\nabla\log p(x)\approx(\mu_1-x)/\sigma^2=(-3+2.5)/0.25=-2$: the point is pulled to the left, towards the nearer mean $-3$. Far from both means the arrow always points to the *nearer* component, which is why plain Langevin keeps each chain in its starting basin.</details>
4. Why did plain Langevin dynamics report 50 / 50 for an 80 / 20 mixture, and why does annealing help?
   <details><summary>Answer</summary>Each chain is trapped in the basin it starts in, because the valley between the states has almost zero density and a small-step chain essentially never crosses it; proportions then follow the starting points. A heavily noise-blurred density has a single broad hill whose slopes reflect the 80 / 20 masses, so chains settle in the right proportions before the noise is reduced.</details>
5. Write the classifier-free guided velocity. What do $\gamma=0$, $\gamma=1$ and $\gamma=5$ do, and what did Section 6 measure for recall?
   <details><summary>Answer</summary>$v_\gamma=v(x,t,\varnothing)+\gamma[v(x,t,y)-v(x,t,\varnothing)]$. $\gamma=0$: unconditional (≈ 1/6 of cells of the requested type); $\gamma=1$: the conditional model (≈ 99 %, matching the classifier's accuracy on real cells); $\gamma=5$: exaggerated, 100 % correct type but recall dropped from 0.76 to 0.32 — less diversity, distorted marker genes.</details>
6. How many network evaluations does it cost to generate one sample with (a) DDPM with 1,000 steps, (b) flow matching with 20 Euler steps, (c) guided flow matching with 40 steps?
   <details><summary>Answer</summary>(a) 1,000; (b) 20; (c) 80, because each guided step evaluates the network twice (with and without the label).</details>
7. Code reading: in `train_flow_matching`, what would happen with `p_drop=0` and why could guidance then not be used?
   <details><summary>Answer</summary>The label is never replaced by the 'no label' token, so the network never learns the unconditional field $v(x,t,\varnothing)$ — the embedding of that token stays untrained — and $v_\gamma$ would combine the conditional field with a meaningless one.</details>
8. Why does the closed-form minimiser of the flow-matching loss on a finite training set memorise, while the neural network did not?
   <details><summary>Answer</summary>Its velocity points to a softmax-weighted average of training points whose weights become one-hot as $t\to1$, so every trajectory ends exactly on a training point. The network cannot represent this extremely sharp field with limited capacity and training; its smooth approximation spreads mass between and around the training points — its inductive bias is what generalises.</details>

## 🧠 Questions & discussion

1. Section 6 showed that guidance makes generated cells 'more typical than real'. If a generative model were used to augment a rare-disease data set or to propose candidate cell states for an experiment, which value of γ would you choose, and how would you check that rare states are not being erased?
2. A diffusion model trained on patient single-cell profiles is shared publicly as a 'privacy-preserving' substitute for the data. Using Section 8, explain the risk, how you would test for it, and what safeguards (e.g. differential privacy, access control) you would ask for.
3. RFdiffusion builds rotation equivariance into its denoiser; AlphaFold 3 learns it from augmented data. What are the costs and benefits of each choice for data efficiency, flexibility and speed?
4. Diffusion models for proteins can design binders to almost any target. Which designs raise dual-use concerns, and what roles should screening of DNA synthesis orders, model access policies and community commitments play?
5. Energy distance, precision/recall, FID and designability each capture part of 'quality'. Propose an evaluation protocol for a generator of perturbation responses that a sceptical biologist would accept.

## 🚀 Bonus — going further

- **DDIM.** Implement the deterministic DDIM update $x_{t'}=\sqrt{\bar\alpha_{t'}}\,\hat x_0+\sqrt{1-\bar\alpha_{t'}}\,\varepsilon_\theta$ with $\hat x_0=(x_t-\sqrt{1-\bar\alpha_t}\varepsilon_\theta)/\sqrt{\bar\alpha_t}$ using the trained `ddpm_fork`, and add its curve to the quality-vs-steps plot of 5.2.
- **Minibatch optimal-transport pairing.** In `train_flow_matching`, re-pair each batch of noise and data points by solving a small assignment problem (`scipy.optimize.linear_sum_assignment` on squared distances) before building the paths. Are the paths straighter and the 5-step samples better (Tong *et al.* 2024)?
- **Conditional generation of perturbation responses.** Simulate control and stimulated cells with `simulate_scrna` (two donors per condition), condition the flow on (cell type, condition), hold out *stimulated monocytes* during training, and ask the model to generate them. Compare with the real held-out cells — and with the simple baseline 'control monocytes + average stimulation shift of the other types'.
- **A tiny backbone generator.** Train a DDPM on 2,000 noisy, randomly rotated copies of the helix bundle (`simulate_helix_bundle(rng=rng, noise=0.3)`), first with a plain MLP on flattened coordinates and then with the EGNN layer of [Module 28](28_graph_neural_networks.ipynb). Which generates valid 3.8 Å spacing and helices?
- **Try a production model.** In a GPU Colab, run an open protein diffusion model (e.g. RFdiffusion through ColabDesign) to generate a few backbones, and score them with the TM-score and lDDT code of [Module 38](38_protein_structure.ipynb).

## 📝 Lab note

Copy this template into your lab notebook and fill it in.

```text
Module: 27 · Generative models: diffusion and flow matching
Biological question: can a model learn a distribution of cells (or structures) well enough to generate new,
                     realistic, diverse and requested examples?
Data (simulated): sample_fork (2-D fork; true curves, width 0.12, shares 50/30/20 %); two-state Gaussian mixture
                  (exact score); simulate_scrna PBMC-like cells (6 true types, marker genes) in 20-D PCA space;
                  simulate_helix_bundle Cα coordinates
Method: Langevin and annealed Langevin; DDPM (linear schedule, T = 1000, ε-prediction MLP, ancestral sampling);
        flow matching (straight paths, Euler); class-conditional flow with label dropout + classifier-free guidance
Key equation: q(x_t | x_0) = N(sqrt(ᾱ_t) x_0, (1 − ᾱ_t) I);  L = ||ε − ε_θ(x_t, t)||²;  s ≈ −ε_θ / sqrt(1 − ᾱ_t)
              L_FM = ||v_θ(x_t, t) − (x_1 − x_0)||²;  v_γ = v_∅ + γ (v_y − v_∅)
Key parameters: schedule, T, network width, training steps, sampling steps (NFE), label-dropout rate, guidance γ
Ground-truth check result: (score cosine; energy distance vs floor; branch shares; requested-type % / precision /
                           recall vs γ; marker log-FC; nearest-neighbour memorisation test)
What changed when I modified … (sampling steps / γ / training-set size / schedule): …
One failure mode: (e.g. plain Langevin gets mode proportions wrong; large γ erases rare states; perfect fit = memoriser)
Connection to research (paper/tool): DDPM (Ho et al. 2020); flow matching (Lipman et al. 2023); CFG (Ho & Salimans
                                     2022); RFdiffusion; AlphaFold 3
```

## 📚 References

- Sohl-Dickstein J., Weiss E., Maheswaranathan N., Ganguli S. (2015) Deep unsupervised learning using nonequilibrium thermodynamics. *Proc ICML*.
- Ho J., Jain A., Abbeel P. (2020) Denoising diffusion probabilistic models. *NeurIPS*.
- Song Y., Ermon S. (2019) Generative modeling by estimating gradients of the data distribution. *NeurIPS*.
- Song Y., Sohl-Dickstein J., Kingma D. P., Kumar A., Ermon S., Poole B. (2021) Score-based generative modeling through stochastic differential equations. *ICLR*.
- Vincent P. (2011) A connection between score matching and denoising autoencoders. *Neural Comput* 23:1661–1674.
- Nichol A., Dhariwal P. (2021) Improved denoising diffusion probabilistic models. *Proc ICML*.
- Dhariwal P., Nichol A. (2021) Diffusion models beat GANs on image synthesis. *NeurIPS*.
- Song J., Meng C., Ermon S. (2021) Denoising diffusion implicit models. *ICLR*.
- Ho J., Salimans T. (2022) Classifier-free diffusion guidance. arXiv:2207.12598.
- Karras T., Aittala M., Aila T., Laine S. (2022) Elucidating the design space of diffusion-based generative models. *NeurIPS*.
- Song Y., Dhariwal P., Chen M., Sutskever I. (2023) Consistency models. *Proc ICML*.
- Lipman Y., Chen R. T. Q., Ben-Hamu H., Nickel M., Le M. (2023) Flow matching for generative modeling. *ICLR*.
- Liu X., Gong C., Liu Q. (2023) Flow straight and fast: learning to generate and transfer data with rectified flow. *ICLR*.
- Albergo M. S., Vanden-Eijnden E. (2023) Building normalizing flows with stochastic interpolants. *ICLR*.
- Tong A. *et al.* (2024) Improving and generalizing flow-based generative models with minibatch optimal transport. *Trans Mach Learn Res*.
- Chen R. T. Q., Lipman Y. (2024) Flow matching on general geometries. *ICLR*.
- Esser P. *et al.* (2024) Scaling rectified flow transformers for high-resolution image synthesis. *Proc ICML*.
- Goodfellow I. *et al.* (2014) Generative adversarial nets. *NeurIPS*.
- Dinh L., Sohl-Dickstein J., Bengio S. (2017) Density estimation using Real NVP. *ICLR*.
- Roberts G. O., Tweedie R. L. (1996) Exponential convergence of Langevin distributions and their discrete approximations. *Bernoulli* 2:341–363.
- Székely G. J., Rizzo M. L. (2013) Energy statistics: a class of statistics based on distances. *J Stat Plan Inference* 143:1249–1272.
- Gretton A., Borgwardt K. M., Rasch M. J., Schölkopf B., Smola A. (2012) A kernel two-sample test. *J Mach Learn Res* 13:723–773.
- Kynkäänniemi T., Karras T., Laine S., Lehtinen J., Aila T. (2019) Improved precision and recall metric for assessing generative models. *NeurIPS*.
- Carlini N. *et al.* (2023) Extracting training data from diffusion models. *Proc USENIX Security Symposium*.
- Austin J., Johnson D. D., Ho J., Tarlow D., van den Berg R. (2021) Structured denoising diffusion models in discrete state-spaces. *NeurIPS*.
- Watson J. L. *et al.* (2023) De novo design of protein structure and function with RFdiffusion. *Nature* 620:1089–1100.
- Ingraham J. B. *et al.* (2023) Illuminating protein space with a programmable generative model. *Nature* 623:1070–1078.
- Abramson J. *et al.* (2024) Accurate structure prediction of biomolecular interactions with AlphaFold 3. *Nature* 630:493–500.
- Corso G., Stärk H., Jing B., Barzilay R., Jaakkola T. (2023) DiffDock: diffusion steps, twists, and turns for molecular docking. *ICLR*.
- Hoogeboom E., Satorras V. G., Vignac C., Welling M. (2022) Equivariant diffusion for molecule generation in 3D. *Proc ICML*.
- Krishna R. *et al.* (2024) Generalized biomolecular modeling and design with RoseTTAFold All-Atom. *Science* 384:eadl2528.
- Hayes T. *et al.* (2025) Simulating 500 million years of evolution with a language model. *Science* 387:850–858.
- Lopez R., Regier J., Cole M. B., Jordan M. I., Yosef N. (2018) Deep generative modeling for single-cell transcriptomics. *Nat Methods* 15:1053–1058.
- Lotfollahi M., Wolf F. A., Theis F. J. (2019) scGen predicts single-cell perturbation responses. *Nat Methods* 16:715–721.
- Lotfollahi M. *et al.* (2023) Predicting cellular responses to complex perturbations in high-throughput screens. *Mol Syst Biol* 19:e11517.
- Bunne C. *et al.* (2023) Learning single-cell perturbation responses using neural optimal transport. *Nat Methods* 20:1759–1768.
- Bunne C. *et al.* (2024) How to build the virtual cell with artificial intelligence: priorities and opportunities. *Cell* 187:7045–7063.
- Ahlmann-Eltze C., Huber W., Anders S. (2025) Deep-learning-based gene perturbation effect prediction does not yet outperform simple linear baselines. *Nature Methods*.
- Kang H. M. *et al.* (2018) Multiplexed droplet single-cell RNA-sequencing using natural genetic variation. *Nat Biotechnol* 36:89–94.
- Replogle J. M. *et al.* (2022) Mapping information-rich genotype-phenotype landscapes with genome-scale Perturb-seq. *Cell* 185:2559–2575.

## ➡️ Next

Diffusion models need denoisers that respect the geometry of molecules. **[Module 28 · Graph neural networks and
geometric deep learning](28_graph_neural_networks.ipynb)** builds message passing on graphs and an equivariant layer
for 3-D point clouds. On the proteins track, **[Module 38 · Protein structure](38_protein_structure.ipynb)** shows
how structures are compared and predicted, and **[Module 39 · Generative design](39_generative_design.ipynb)** puts
generative models to work designing proteins and molecules.